# Short-term electricity demand forecasting for New Zealand: full pipeline

IT8105 Machine Learning, Assessment 2. This notebook holds the whole code base in one file. It combines the shared modules (`common.py`, `deep.py`) and the 17 numbered step scripts in `code/`, in the run order given in the README. Each step section has the same code as its script. The only changes are the ones needed to run in one notebook:

- the `from common import *` and `from deep import *` lines are dropped, because those modules are defined in the first two code cells;
- steps 05 and 07 loop over `LSTM` and `Transformer` instead of reading the model name from the command line;
- step 12 no longer launches `11_cv_figures.py` as a subprocess, because step 11 runs in the cell just before it;
- `ROOT` is the repository root (the folder this notebook sits in) instead of being worked out from the script path.

**Data.** The data are not included. Put the 24 EMI `YYYYMM_Grid_export.csv` files and the three Open-Meteo city files in `data/`, as described in the README.

**Running time.** A full run takes about 3 hours on an Apple-silicon MacBook Pro (CPU only), mostly LSTM tuning (step 05) and ablations (step 07). Results, figures and models go to `out/`. Steps 09 to 17 only read saved outputs (apart from step 16, which retrains the selected Transformer), so they can be rerun on their own once steps 01 to 08 have finished.

**macOS note.** LightGBM (Homebrew `libomp`) and PyTorch (its own bundled `libomp`) can crash with a segmentation fault when both load in one process, and in a notebook they always share one kernel. Start Jupyter from a terminal with the loader pointed at PyTorch's copy:

    export DYLD_LIBRARY_PATH="$(python -c 'import torch, os; print(os.path.join(os.path.dirname(torch.__file__), "lib"))')"
    jupyter notebook

**Environment.** Python 3.13, pandas 3.0.5, numpy 2.5.3, scikit-learn 1.9.1, lightgbm 4.7.0, torch 2.14.1 (CPU), statsmodels 0.15.0, shap 0.54.1, holidays 0.107, matplotlib 3.11.2 (see `requirements.txt`). Random seed 42; the neural networks are repeated with seeds 42 to 46.

## Shared utilities (`common.py`)
Paths, plotting style, feature building, rolling-origin folds, targets and peak weights, metrics, the Diebold-Mariano test and the day-block bootstrap.

In [ ]:
"""Shared utilities: paths, plotting style, features, folds, metrics, statistics."""
import os, json, time
import numpy as np, pandas as pd

# repository root: the folder this notebook is in (or its parent if the notebook is started from code/)
ROOT = os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == "code" else os.getcwd()
PROC = os.path.join(ROOT, "data", "processed")
FIG = os.path.join(ROOT, "out", "figs"); TAB = os.path.join(ROOT, "out", "tables"); MOD = os.path.join(ROOT, "out", "models")
for p in (FIG, TAB, MOD): os.makedirs(p, exist_ok=True)
SEED = 42
FIRST_TARGET_DAY = "2024-09-08"          # first day with a full 7-day history
TEST_START, TEST_END = "2026-06-01", "2026-08-31"
# expanding-window rolling-origin folds: validation blocks of two months
FOLDS = [("2025-06-01", "2025-07-31"), ("2025-08-01", "2025-09-30"), ("2025-10-01", "2025-11-30"),
         ("2025-12-01", "2026-01-31"), ("2026-02-01", "2026-03-31"), ("2026-04-01", "2026-05-31")]
HDD_BASE = 16.0
PW_LAMBDA = 3.0                            # Bansal (2026): weight 1 + 2 = 3 on peak samples
COLORS = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#6250d6", "#e34948"]
INK, MUTED = "#0b0b0b", "#52514e"

def set_style():
    import matplotlib as mpl
    mpl.rcParams.update({"font.family": "serif", "font.serif": ["Liberation Serif", "Times New Roman", "Times", "DejaVu Serif"],
        "font.size": 9, "axes.titlesize": 10, "axes.labelsize": 9, "legend.fontsize": 8,
        "xtick.labelsize": 8, "ytick.labelsize": 8, "axes.spines.top": False, "axes.spines.right": False,
        "axes.edgecolor": MUTED, "axes.labelcolor": INK, "xtick.color": MUTED, "ytick.color": MUTED,
        "axes.grid": True, "grid.color": "#e6e5e0", "grid.linewidth": 0.6, "lines.linewidth": 1.5,
        "figure.dpi": 150, "savefig.dpi": 300, "savefig.bbox": "tight", "legend.frameon": False})

def savefig(fig, name):
    fig.savefig(os.path.join(FIG, name + ".png")); fig.savefig(os.path.join(FIG, name + ".pdf"))

def load_grid():
    return pd.read_pickle(os.path.join(PROC, "national_halfhourly.pkl"))

# ------------------------------------------------------------------ features (tabular)
WEATHER_COLS = {"popw": "temp_popw", "avg3": "temp_avg3", "akl": "temp_auc"}

def day_matrix(g, col):
    """Return DataFrame indexed by date with 48 columns for a given column."""
    return g.pivot(index="date", columns="slot", values=col)

def build_tabular(g, weather="observed", city="popw"):
    """One row per (target day D, slot s). Origin = end of day D-1 (all of D-1 observed).
    weather: 'observed' (realised weather at target time, an oracle for a day-ahead forecast),
             'persist' (same slot of day D-1, available at forecast time), 'none'."""
    Y = day_matrix(g, "mw"); T = day_matrix(g, WEATHER_COLS[city])
    W = (day_matrix(g, "wind_auc") + day_matrix(g, "wind_wel") + day_matrix(g, "wind_chr")) / 3
    hol = g.groupby("date").holiday.first()
    dates = Y.index.tolist(); di = {d: i for i, d in enumerate(dates)}
    Yv, Tv, Wv = Y.values, T.values, W.values
    recs = []
    for d in dates:
        i = di[d]
        if d < FIRST_TARGET_DAY: continue
        dt = pd.Timestamp(d)
        y1, y2, y7 = Yv[i - 1], Yv[i - 2], Yv[i - 7]
        past7 = Yv[i - 7:i]
        if weather == "observed": tt, tw = Tv[i], Wv[i]
        elif weather == "persist": tt, tw = Tv[i - 1], Wv[i - 1]
        else: tt, tw = None, None
        for s in range(48):
            r = dict(date=d, slot=s, y=Yv[i, s], origin=y1[47],
                     slot_sin=np.sin(2 * np.pi * s / 48), slot_cos=np.cos(2 * np.pi * s / 48),
                     dow=dt.dayofweek, weekend=int(dt.dayofweek >= 5), holiday=int(hol[d]),
                     hol_prev=int(hol[dates[i - 1]]), hol_next=int(hol[dates[i + 1]]) if i + 1 < len(dates) else 0,
                     doy_sin=np.sin(2 * np.pi * dt.dayofyear / 365.25), doy_cos=np.cos(2 * np.pi * dt.dayofyear / 365.25),
                     lag_d1=y1[s], lag_d2=y2[s], lag_d7=y7[s], d1_mean=y1.mean(), d1_max=y1.max(),
                     w7_mean_slot=past7[:, s].mean(), w7_max=past7.max(), d1_min=y1.min())
            if tt is not None:
                r.update(temp=tt[s], hdd=max(0.0, HDD_BASE - tt[s]), temp_dmean=tt.mean(), temp_dmin=tt.min(),
                         wind=tw[s], temp_lag_d1=Tv[i - 1][s])
            recs.append(r)
    return pd.DataFrame(recs)

FEATURE_GROUPS = {
    "calendar": ["slot", "slot_sin", "slot_cos", "dow", "weekend", "holiday", "hol_prev", "hol_next", "doy_sin", "doy_cos"],
    "lags": ["origin", "lag_d1", "lag_d2", "lag_d7", "d1_mean", "d1_max", "d1_min", "w7_mean_slot", "w7_max"],
    "weather": ["temp", "hdd", "temp_dmean", "temp_dmin", "wind", "temp_lag_d1"],
}

def feature_cols(df, drop=()):
    cols = []
    for gname, gc in FEATURE_GROUPS.items():
        if gname in drop: continue
        cols += [c for c in gc if c in df.columns]
    return cols

# ------------------------------------------------------------------ targets & weights
def make_target(df, kind):
    if kind == "abs": return df["y"].values, np.zeros(len(df))
    if kind == "delta_origin": return (df["y"] - df["origin"]).values, df["origin"].values
    if kind == "delta_seasonal": return (df["y"] - df["lag_d1"]).values, df["lag_d1"].values
    raise ValueError(kind)

def make_weights(y_abs, scheme, ref=None, lam=None):
    """Peak weights computed on absolute demand, not on the delta target. ref = training-set demand used for thresholds."""
    ref = y_abs if ref is None else ref; lam = PW_LAMBDA if lam is None else lam
    if scheme == "none": return np.ones(len(y_abs))
    if scheme == "bansal": return 1 + (lam - 1) * (y_abs > np.mean(ref))
    if scheme == "topdecile": return 1 + (lam - 1) * (y_abs >= np.quantile(ref, 0.9))
    raise ValueError(scheme)

def split_masks(df, fold=None, test=False):
    d = df["date"]
    if test:
        return (d < TEST_START).values, ((d >= TEST_START) & (d <= TEST_END)).values
    vs, ve = FOLDS[fold]
    return (d < vs).values, ((d >= vs) & (d <= ve)).values

# ------------------------------------------------------------------ metrics
def metrics(y, p, dates=None, slots=None):
    y = np.asarray(y, float); p = np.asarray(p, float); e = y - p; ae = np.abs(e)
    thr = np.quantile(y, 0.9); top = y >= thr
    w = np.where(top, PW_LAMBDA, 1.0)
    out = dict(MAE=ae.mean(), RMSE=np.sqrt((e ** 2).mean()), MAPE=100 * np.mean(ae / y),
               R2=1 - (e ** 2).sum() / ((y - y.mean()) ** 2).sum(),
               PW_MAE=np.mean(w * ae), TD_MAE=ae[top].mean(), TD_under=100 * np.mean(p[top] < y[top]),
               bias=np.mean(p - y))
    if dates is not None:
        dfp = pd.DataFrame(dict(d=dates, s=slots, y=y, p=p))
        a = dfp.pivot(index="d", columns="s", values="y").values; b = dfp.pivot(index="d", columns="s", values="p").values
        out["PeakMAE"] = np.mean(np.abs(a.max(1) - b.max(1)))
        out["PeakTimeErr_h"] = np.mean(np.abs(a.argmax(1) - b.argmax(1))) / 2
        out["PeakUnder"] = 100 * np.mean(b.max(1) < a.max(1))
    return out

def dm_test(e1, e2, h=1, power=1):
    """Diebold-Mariano test with Harvey-Leybourne-Newbold small-sample correction.
    e1, e2: forecast errors of model 1 and 2 on the same targets (here: daily mean abs errors).
    Returns (DM statistic, two-sided p-value). Positive DM => model 1 has larger loss."""
    from scipy import stats
    d = np.abs(e1) ** power - np.abs(e2) ** power; n = len(d); dbar = d.mean()
    gam = [np.sum((d[k:] - dbar) * (d[:n - k] - dbar)) / n for k in range(h)]
    var = (gam[0] + 2 * sum(gam[1:])) / n
    dm = dbar / np.sqrt(var)
    hln = np.sqrt((n + 1 - 2 * h + h * (h - 1) / n) / n)
    stat = dm * hln
    return stat, 2 * stats.t.sf(np.abs(stat), df=n - 1)

def daily_mae(dates, y, p):
    return pd.DataFrame(dict(d=dates, ae=np.abs(np.asarray(y) - np.asarray(p)))).groupby("d").ae.mean()

def block_bootstrap_ci(dates, y, p, fn, n=1000, seed=SEED):
    """Day-block bootstrap CI of a metric: resample whole days with replacement."""
    rng = np.random.default_rng(seed)
    df = pd.DataFrame(dict(d=dates, y=y, p=p)); groups = [g for _, g in df.groupby("d")]
    vals = []
    for _ in range(n):
        idx = rng.integers(0, len(groups), len(groups))
        b = pd.concat([groups[i] for i in idx]); vals.append(fn(b.y.values, b.p.values))
    return np.percentile(vals, [2.5, 97.5])

def save_json(obj, name):
    json.dump(obj, open(os.path.join(TAB, name), "w"), indent=1, default=float)

def pretty(name):
    """'Transformer delta_seasonal-none' -> 'Transformer (Δ-slot, L1)'"""
    rep = {"abs-L1": "(abs, L1)", "delta_seasonal-none": "(Δ-slot, L1)", "delta_seasonal-bansal": "(Δ-slot, PW mean)",
           "delta_seasonal-topdecile": "(Δ-slot, PW top-10%)", "delta_origin-none": "(Δ-origin, L1)", "delta_origin-bansal": "(Δ-origin, PW mean)",
           "delta_origin-topdecile": "(Δ-origin, PW top-10%)"}
    for k, v in rep.items(): name = name.replace(k, v)
    return name

## Neural-network models (`deep.py`)
Daily sample builder, training-only scaler, the LSTM and patch-Transformer forecasters, and the shared trainer with early stopping.

In [ ]:
"""LSTM and patch-Transformer day-ahead forecasters (PyTorch), shared data builder and trainer."""
import warnings; warnings.filterwarnings("ignore", message="enable_nested_tensor")
import torch, torch.nn as nn
torch.set_num_threads(int(os.environ.get("TORCH_THREADS", 2)))

def build_daily(g, weather="observed", city="popw"):
    """Arrays indexed by target day D. hist: (7*48, Fh) for days D-7..D-1; fut: (48, Ff) for day D."""
    Y = day_matrix(g, "mw").values; T = day_matrix(g, WEATHER_COLS[city]).values
    dates = sorted(g.date.unique()); hol = g.groupby("date").holiday.first().values
    idx = [i for i, d in enumerate(dates) if d >= FIRST_TARGET_DAY]
    s = np.arange(48); ssin, scos = np.sin(2 * np.pi * s / 48), np.cos(2 * np.pi * s / 48)
    H, Fu, Yt, O, L1, D = [], [], [], [], [], []
    for i in idx:
        dt = pd.Timestamp(dates[i]); hrows = []
        for j in range(i - 7, i):
            dj = pd.Timestamp(dates[j])
            hrows.append(np.stack([Y[j], T[j], np.full(48, hol[j]), np.full(48, dj.dayofweek >= 5), ssin, scos], 1))
        H.append(np.concatenate(hrows, 0))
        if weather == "observed": tt = T[i]
        elif weather == "persist": tt = T[i - 1]
        else: tt = np.zeros(48)
        dow = np.zeros((48, 7)); dow[:, dt.dayofweek] = 1
        nxt = hol[i + 1] if i + 1 < len(dates) else 0
        Fu.append(np.column_stack([tt, np.maximum(0, HDD_BASE - tt) if weather != "none" else np.zeros(48), ssin, scos, dow,
                                   np.full(48, hol[i]), np.full(48, hol[i - 1]), np.full(48, nxt),
                                   np.full(48, np.sin(2 * np.pi * dt.dayofyear / 365.25)), np.full(48, np.cos(2 * np.pi * dt.dayofyear / 365.25)),
                                   Y[i - 1], Y[i - 7]]))
        Yt.append(Y[i]); O.append(Y[i - 1, 47]); L1.append(Y[i - 1]); D.append(dates[i])
    return dict(H=np.array(H, np.float32), F=np.array(Fu, np.float32), Y=np.array(Yt, np.float32),
                O=np.array(O, np.float32), L1=np.array(L1, np.float32), dates=np.array(D))

class Scaler:
    """Fitted on training days only: demand and temperature standardisation."""
    def fit(self, d, m):
        self.ym, self.ys = d["Y"][m].mean(), d["Y"][m].std()
        self.tm, self.ts = d["F"][m][:, :, 0].mean(), d["F"][m][:, :, 0].std() + 1e-6
        self.hm, self.hs = d["F"][m][:, :, 1].mean(), d["F"][m][:, :, 1].std() + 1e-6
        return self
    def transform(self, d, m):
        H = d["H"][m].copy(); F = d["F"][m].copy()
        H[:, :, 0] = (H[:, :, 0] - self.ym) / self.ys; H[:, :, 1] = (H[:, :, 1] - self.tm) / self.ts
        F[:, :, 0] = (F[:, :, 0] - self.tm) / self.ts; F[:, :, 1] = (F[:, :, 1] - self.hm) / self.hs
        F[:, :, -2:] = (F[:, :, -2:] - self.ym) / self.ys
        return H, F

class Head(nn.Module):
    def __init__(self, zdim, ff, hh, drop):
        super().__init__(); self.net = nn.Sequential(nn.Linear(zdim + ff, hh), nn.GELU(), nn.Dropout(drop), nn.Linear(hh, 1))
    def forward(self, z, F):                      # z: (B, zdim), F: (B, 48, ff)
        return self.net(torch.cat([z.unsqueeze(1).expand(-1, F.shape[1], -1), F], -1)).squeeze(-1)

class LSTMNet(nn.Module):
    def __init__(self, fh, ff, hidden=64, layers=1, dropout=.1, head_hidden=64, **_):
        super().__init__()
        self.lstm = nn.LSTM(fh, hidden, layers, batch_first=True, dropout=dropout if layers > 1 else 0)
        self.head = Head(hidden, ff, head_hidden, dropout)
    def forward(self, H, F):
        # 336 half-hours -> 168 hourly steps (mean of each pair), matching the 168-h lookback of Bansal (2026)
        Hh = H.reshape(H.shape[0], 168, 2, H.shape[-1]).mean(2)
        _, (h, _) = self.lstm(Hh); return self.head(h[-1], F)

class PatchTransformer(nn.Module):
    """Each past day (48 half-hours x Fh) is one token; the target day's known covariates form an 8th token."""
    def __init__(self, fh, ff, d_model=64, heads=4, layers=2, dropout=.1, head_hidden=64, **_):
        super().__init__()
        self.hist_proj = nn.Linear(48 * fh, d_model); self.fut_proj = nn.Linear(48 * ff, d_model)
        self.pos = nn.Parameter(torch.randn(8, d_model) * .02)
        enc = nn.TransformerEncoderLayer(d_model, heads, 2 * d_model, dropout, batch_first=True, norm_first=True, activation="gelu")
        self.enc = nn.TransformerEncoder(enc, layers); self.norm = nn.LayerNorm(d_model)
        self.head = Head(d_model, ff, head_hidden, dropout)
    def forward(self, H, F):
        B = H.shape[0]; tok = torch.cat([self.hist_proj(H.reshape(B, 7, -1)), self.fut_proj(F.reshape(B, 1, -1))], 1) + self.pos
        z = self.norm(self.enc(tok))[:, -1]; return self.head(z, F)

ARCH = {"LSTM": LSTMNet, "Transformer": PatchTransformer}

def fit_predict(d, tr, te, arch, hp, target="abs", weights="none", seed=SEED, max_epochs=120, patience=12, return_model=False):
    torch.manual_seed(seed); np.random.seed(seed)
    tr_idx = np.where(tr)[0]; n_in = max(14, int(round(.1 * len(tr_idx))))
    fit_idx, es_idx = tr_idx[:-n_in], tr_idx[-n_in:]           # chronological inner split for early stopping
    fm = np.zeros(len(tr), bool); fm[fit_idx] = True
    sc = Scaler().fit(d, fm)
    def base(m):
        if target == "abs": return np.zeros_like(d["Y"][m])
        if target == "delta_origin": return np.repeat(d["O"][m][:, None], 48, 1)
        if target == "delta_seasonal": return d["L1"][m]
    def tens(m):
        H, F = sc.transform(d, m); yt = (d["Y"][m] - base(m) - (sc.ym if target == "abs" else 0)) / sc.ys
        w = make_weights(d["Y"][m].ravel(), weights, ref=d["Y"][fm].ravel()).reshape(-1, 48)
        return [torch.tensor(a, dtype=torch.float32) for a in (H, F, yt, w)]
    esm = np.zeros(len(tr), bool); esm[es_idx] = True
    Htr, Ftr, Ytr, Wtr = tens(fm); Hes, Fes, Yes, Wes = tens(esm)
    model = ARCH[arch](Htr.shape[-1], Ftr.shape[-1], **hp)
    opt = torch.optim.AdamW(model.parameters(), lr=hp["lr"], weight_decay=1e-4)
    bs = hp.get("batch", 32); best, best_state, bad, hist = np.inf, None, 0, []
    for ep in range(max_epochs):
        model.train(); perm = torch.randperm(len(Htr)); tl = 0
        for b in range(0, len(perm), bs):
            ii = perm[b:b + bs]; opt.zero_grad()
            loss = (Wtr[ii] * (model(Htr[ii], Ftr[ii]) - Ytr[ii]).abs()).mean(); loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step(); tl += loss.item() * len(ii)
        model.eval()
        with torch.no_grad(): vl = (Wes * (model(Hes, Fes) - Yes).abs()).mean().item()
        hist.append((tl / len(Htr), vl))
        if vl < best - 1e-5: best, best_state, bad = vl, {k: v.clone() for k, v in model.state_dict().items()}, 0
        else:
            bad += 1
            if bad >= patience: break
    model.load_state_dict(best_state); model.eval()
    H, F = sc.transform(d, te)
    with torch.no_grad(): out = model(torch.tensor(H), torch.tensor(F)).numpy()
    pred = out * sc.ys + (sc.ym if target == "abs" else 0) + base(te)
    info = dict(epochs=len(hist), best_epoch=int(np.argmin([h[1] for h in hist]) + 1), history=hist,
                n_params=sum(p.numel() for p in model.parameters()))
    return (pred, info, model, sc) if return_model else (pred, info)

def day_masks(d, fold=None, test=False):
    dd = pd.Series(d["dates"])
    if test: return (dd < TEST_START).values, ((dd >= TEST_START) & (dd <= TEST_END)).values
    vs, ve = FOLDS[fold]; return (dd < vs).values, ((dd >= vs) & (dd <= ve)).values

def flat_metrics(d, m, pred):
    y = d["Y"][m]; dates = np.repeat(d["dates"][m], 48); slots = np.tile(np.arange(48), m.sum())
    return metrics(y.ravel(), pred.ravel(), dates, slots)

## Step 01: build the national half-hourly series
`code/01_build_dataset.py`. Merges the EMI files, converts kWh to MW, maps trading periods to true UTC and then to a 48-slot local-clock grid (DST fix), aligns the weather time zones and writes the data audit.

In [ ]:
"""Step 1: build the half-hourly national demand and weather dataset.

Reads the 24 monthly EMI Grid_export files and the three Open-Meteo city files,
and writes data/processed/national_halfhourly.parquet (local-clock 48-slot grid).
Every number about the raw data that the report quotes is printed here and saved
to out/tables/data_audit.json.
"""
import glob, json, os
import numpy as np, pandas as pd, holidays

RAW = os.path.join(ROOT, "data")
OUT = os.path.join(ROOT, "data", "processed"); os.makedirs(OUT, exist_ok=True)
TAB = os.path.join(ROOT, "out", "tables"); os.makedirs(TAB, exist_ok=True)
TZ = "Pacific/Auckland"
audit = {}

# ---------------------------------------------------------------- demand
files = sorted(glob.glob(os.path.join(RAW, "20*_Grid_export.csv")))
raw = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)
tps = [f"TP{i}" for i in range(1, 51)]
audit.update(files=len(files), raw_rows=int(len(raw)), raw_cols=int(raw.shape[1]),
             pocs=int(raw.POC.nunique()), days=int(raw.Trading_Date.nunique()),
             exact_duplicate_rows=int(raw.duplicated().sum()),
             duplicate_keys=int(raw.duplicated(["POC", "Nwk_Code", "Trader", "Trading_Date"]).sum()),
             negative_values=int((raw[tps] < 0).sum().sum()),
             unit=raw.Unit_Measure.unique().tolist(), flow=raw.Flow_Direction.unique().tolist(),
             status=raw.Status.unique().tolist())
pocs_per_month = raw.assign(m=raw.Trading_Date.str[:7]).groupby("m").POC.nunique()
audit["pocs_per_month_min"], audit["pocs_per_month_max"] = int(pocs_per_month.min()), int(pocs_per_month.max())

# national total per trading date and TP (kWh) -> MW (kWh / 500)
nat = raw.groupby("Trading_Date")[tps].sum(min_count=1)
tiwai = raw[raw.POC == "TWI2201"].groupby("Trading_Date")[tps].sum(min_count=1)
n_tp = nat.notna().sum(axis=1)
audit["days_by_tp_count"] = n_tp.value_counts().sort_index().to_dict()
audit["dst_days"] = {d: int(n) for d, n in n_tp[n_tp != 48].items()}

rows = []
for d, r in nat.iterrows():
    n = int(n_tp[d])
    midnight_utc = pd.Timestamp(d).tz_localize(TZ).tz_convert("UTC")
    for k in range(1, n + 1):          # TPs count real elapsed half-hours from local midnight
        rows.append((midnight_utc + pd.Timedelta(minutes=30 * (k - 1)), d, k,
                     r[f"TP{k}"] / 500.0, tiwai.loc[d, f"TP{k}"] / 500.0))
s = pd.DataFrame(rows, columns=["utc", "trading_date", "tp", "mw", "tiwai_mw"])
assert s.utc.is_unique and (s.utc.diff().dropna() == pd.Timedelta("30min")).all()
audit["halfhours_true_time"] = int(len(s))

# map true time onto a 48-slot local-clock grid
s["local"] = s.utc.dt.tz_convert(TZ)
s["date"] = s.local.dt.strftime("%Y-%m-%d")
s["slot"] = s.local.dt.hour * 2 + s.local.dt.minute // 30          # 0..47
g = s.groupby(["date", "slot"], as_index=False)[["mw", "tiwai_mw"]].mean()  # fall-back hour averaged
full = pd.MultiIndex.from_product([sorted(s.date.unique()), range(48)], names=["date", "slot"])
g = g.set_index(["date", "slot"]).reindex(full)
audit["slots_interpolated_spring_forward"] = int(g.mw.isna().sum())
g = g.interpolate(limit=2)                                          # 2 missing slots per spring-forward day
g = g.reset_index()
g["ts"] = pd.to_datetime(g.date) + pd.to_timedelta(g.slot * 30, unit="min")  # naive local clock
audit["grid_rows"] = int(len(g))

# ---------------------------------------------------------------- weather
# NOTE: the Auckland file is in fixed UTC+12, the Wellington and Christchurch files are in UTC.
offsets = {"Auckland": 12, "Wellington": 0, "Christchurch": 0}
wx = {}
for city, off in offsets.items():
    w = pd.read_csv(os.path.join(RAW, f"{city}_weather_2024-09_2026-08.csv"), skiprows=3)
    w.columns = ["time", "temp", "precip", "wind"]
    w["utc"] = (pd.to_datetime(w.time) - pd.Timedelta(hours=off)).dt.tz_localize("UTC")
    w = w.set_index("utc")[["temp", "precip", "wind"]]
    audit[f"wx_{city}_missing"] = int(w.isna().sum().sum())
    wx[city] = w
utc_index = pd.date_range(g.ts.min().tz_localize(TZ).tz_convert("UTC") - pd.Timedelta(hours=1),
                          s.utc.max() + pd.Timedelta(hours=1), freq="30min")
for city, w in wx.items():
    w = w.reindex(w.index.union(utc_index)).sort_index()
    w["temp"] = w["temp"].interpolate(method="time", limit_direction="both")
    w["wind"] = w["wind"].interpolate(method="time", limit_direction="both")
    w["precip"] = w["precip"].ffill().bfill() / 2.0          # hourly mm -> per half-hour
    wx[city] = w.reindex(utc_index)
# local-clock key for weather: average duplicates at fall-back, interpolate spring gap (as for demand)
def to_local_grid(df):
    df = df.copy(); loc = df.index.tz_convert(TZ)
    df["ts"] = loc.tz_localize(None).floor("30min")
    return df.groupby("ts").mean()
for city in offsets:
    lg = to_local_grid(wx[city]).reindex(g.ts).interpolate(limit_direction="both")
    for c in ["temp", "precip", "wind"]:
        g[f"{c}_{city[:3].lower()}"] = lg[c].values

# population weights: Stats NZ 2023 Census regional counts (Auckland, Wellington, Canterbury regions)
POP = {"auc": 1656486, "wel": 520971, "chr": 651027}
wsum = sum(POP.values())
g["temp_avg3"] = g[["temp_auc", "temp_wel", "temp_chr"]].mean(axis=1)
g["temp_popw"] = sum(g[f"temp_{k}"] * v / wsum for k, v in POP.items())

# ---------------------------------------------------------------- calendar
nz = holidays.country_holidays("NZ", years=[2024, 2025, 2026])
g["holiday"] = pd.to_datetime(g.date).dt.date.map(lambda d: d in nz).astype(int)
audit["holidays_in_window"] = sorted({str(d) for d in pd.to_datetime(g.date).dt.date if d in nz})
g.to_pickle(os.path.join(OUT, "national_halfhourly.pkl"))

# headline numbers (half-hourly, local grid)
audit["mw_mean"], audit["mw_sd"] = round(g.mw.mean(), 1), round(g.mw.std(), 1)
audit["mw_min"], audit["mw_max"] = round(g.mw.min(), 1), round(g.mw.max(), 1)
audit["tiwai_mean"] = round(g.tiwai_mw.mean(), 1)
json.dump(audit, open(os.path.join(TAB, "data_audit.json"), "w"), indent=1, default=str)
print(json.dumps(audit, indent=1, default=str))

## Step 02: exploratory data analysis
`code/02_eda.py`. Reruns the EDA on the cleaned, DST-corrected data and draws the EDA figures.

In [ ]:
"""Step 2: re-run the EDA on the processed (local-clock, DST-corrected) data."""
import numpy as np, pandas as pd, matplotlib.pyplot as plt
set_style(); g = load_grid(); res = {}
g["dt"] = pd.to_datetime(g.date); g["month"] = g.dt.dt.month
g["winter"] = g.month.isin([6, 7, 8]); g["split"] = np.where(g.date >= TEST_START, "test", "train")
res["overall"] = g.mw.describe().round(1).to_dict()
res["by_split"] = g.groupby("split").mw.describe().round(1).to_dict(orient="index")
res["winter_mean"], res["nonwinter_mean"] = g[g.winter].mw.mean(), g[~g.winter].mw.mean()
res["weekday_mean"], res["weekend_mean"] = g[g.dt.dt.dayofweek < 5].mw.mean(), g[g.dt.dt.dayofweek >= 5].mw.mean()
res["max_ts"] = str(g.loc[g.mw.idxmax(), "ts"]); res["min_ts"] = str(g.loc[g.mw.idxmin(), "ts"])
dp = g.groupby("date").apply(lambda x: x.slot.values[x.mw.values.argmax()], include_groups=False)
res["modal_peak_slot"] = int(dp.mode()[0]); res["peak_slot_share_evening"] = float(np.mean((dp >= 34) & (dp <= 39)))
res["share_peak_in_winter_top_decile"] = float(g[g.mw >= g.mw.quantile(.9)].winter.mean())
# year-on-year comparison (same months)
yoy = g.assign(yr=np.where(g.dt < "2025-09-01", "Sep24-Aug25", "Sep25-Aug26")).groupby("yr").mw.agg(["mean", "max"])
res["yoy"] = yoy.round(1).to_dict(orient="index")
# temperature correlations: corrected alignment vs naive file-row alignment used in the proposal (Assessment 1)
h = g.assign(hr=g.ts.dt.floor("h")).groupby("hr")[["mw", "temp_avg3", "temp_popw", "temp_auc"]].mean()
res["r_temp_avg3_corrected"] = h.mw.corr(h.temp_avg3); res["r_temp_popw"] = h.mw.corr(h.temp_popw); res["r_temp_akl"] = h.mw.corr(h.temp_auc)
dm = g.groupby("date")[["mw", "temp_popw"]].mean(); res["r_daily_mean_temp_popw"] = dm.mw.corr(dm.temp_popw)
raw = {}
for c in ["Auckland", "Wellington", "Christchurch"]:
    w = pd.read_csv(os.path.join(ROOT, "data", f"{c}_weather_2024-09_2026-08.csv"), skiprows=3); raw[c] = w.iloc[:, 1].values
naive = (raw["Auckland"] + raw["Wellington"] + raw["Christchurch"]) / 3
hh = h.mw.values[: len(naive)]
res["r_temp_naive_A1_alignment"] = float(np.corrcoef(hh, naive[: len(hh)])[0, 1])
# lag correlation of half-hourly demand
y = g.mw.values
res["acf"] = {k: float(np.corrcoef(y[k:], y[:-k])[0, 1]) for k in [1, 2, 48, 96, 336]}
# Tiwai
res["tiwai_mean"], res["tiwai_min"], res["tiwai_max"] = g.tiwai_mw.mean(), g.tiwai_mw.min(), g.tiwai_mw.max()
res["tiwai_share"] = g.tiwai_mw.sum() / g.mw.sum()
td = g.groupby("date").tiwai_mw.mean(); res["tiwai_days_below_450"] = int((td < 450).sum())
# anomaly screen: daily abs change vs same weekday last week
dm = g.groupby("date").mw.mean(); wk = (dm - dm.shift(7)).dropna(); z = (wk - wk.median()) / (1.4826 * (wk - wk.median()).abs().median())
res["anomalous_days_|z|>4"] = {d: round(v, 1) for d, v in z[abs(z) > 4].items()}
# demand vs temperature curve (binned) to choose HDD base
b = h.assign(tb=(h.temp_popw // 1)).groupby("tb").mw.mean(); res["binned_temp_curve"] = b.round(0).to_dict()
save_json(res, "eda.json"); print(json.dumps(res, indent=1, default=float))

# ---- figures
fig, ax = plt.subplots(figsize=(6.6, 2.4))
dd = g.groupby("dt").mw.mean(); ax.plot(dd.index, dd.values, color=COLORS[0], lw=1)
ax.axvspan(pd.Timestamp(TEST_START), pd.Timestamp(TEST_END), color="#e6e5e0", zorder=0)
for vs, ve in FOLDS: ax.axvline(pd.Timestamp(vs), color=MUTED, lw=.6, ls=":")
ax.text(pd.Timestamp("2026-06-05"), dd.max() * .995, "test (winter 2026)", fontsize=7, color=MUTED, va="top")
ax.set_ylabel("Daily mean demand (MW)"); savefig(fig, "fig_eda_daily"); plt.close(fig)

fig, axs = plt.subplots(1, 2, figsize=(6.6, 2.5))
prof = g.groupby(["winter", "slot"]).mw.mean().unstack(0)
x = np.arange(48) / 2
axs[0].plot(x, prof[False], color=COLORS[0], label="Sep–May"); axs[0].plot(x, prof[True], color=COLORS[1], label="Jun–Aug (winter)")
axs[0].set_xlabel("Local time (h)"); axs[0].set_ylabel("Mean demand (MW)"); axs[0].legend(); axs[0].set_xticks(range(0, 25, 6))
axs[1].scatter(h.temp_popw, h.mw, s=1, alpha=.15, color=COLORS[0], rasterized=True)
bb = h.assign(tb=(h.temp_popw // 1 + .5)).groupby("tb").mw.mean(); axs[1].plot(bb.index, bb.values, color=COLORS[1], lw=2, label="1 °C bin mean")
axs[1].axvline(HDD_BASE, color=MUTED, ls="--", lw=.8); axs[1].set_xlabel("Population-weighted temperature (°C)"); axs[1].set_ylabel("Hourly demand (MW)"); axs[1].legend()
fig.tight_layout(); savefig(fig, "fig_eda_profile_temp"); plt.close(fig)

fig, ax = plt.subplots(figsize=(4.2, 2.3))
lags = np.arange(1, 337); ac = [np.corrcoef(y[k:], y[:-k])[0, 1] for k in lags]
ax.plot(lags / 2, ac, color=COLORS[0]); ax.set_xlabel("Lag (hours)"); ax.set_ylabel("Autocorrelation"); ax.set_xticks([0, 24, 48, 72, 96, 120, 144, 168])
savefig(fig, "fig_eda_acf"); plt.close(fig)

## Step 03: feature sets
`code/03_build_tabular.py`. Builds and caches the tabular feature sets (one row per target day and slot) and the daily arrays for the neural networks.

In [ ]:
"""Step 3: build and cache the tabular (one row per target day x slot) feature sets."""
g = load_grid()
for w, c in [("observed", "popw"), ("persist", "popw"), ("none", "popw"), ("observed", "avg3"), ("observed", "akl")]:
    t0 = time.time(); df = build_tabular(g, weather=w, city=c)
    df.to_pickle(os.path.join(PROC, f"tab_{w}_{c}.pkl")); print(w, c, df.shape, round(time.time() - t0, 1), "s")
df = pd.read_pickle(os.path.join(PROC, "tab_observed_popw.pkl"))
info = dict(rows=len(df), days=df.date.nunique(), first=df.date.min(), last=df.date.max(),
            n_features=len(feature_cols(df)), features=feature_cols(df),
            train_days=int((df.date < TEST_START).sum() / 48), test_days=int((df.date >= TEST_START).sum() / 48),
            folds=[dict(val=f, train_days=int((df.date < f[0]).sum() / 48), val_days=int(((df.date >= f[0]) & (df.date <= f[1])).sum() / 48)) for f in FOLDS],
            nan=int(df[feature_cols(df)].isna().sum().sum()))
save_json(info, "tabular_info.json"); print(info)
# daily arrays for the neural networks (one sample per target day)
import pickle
pickle.dump(build_daily(g), open(os.path.join(PROC, "daily_observed_popw.pkl"), "wb"))

## Step 04: tune the baselines, Ridge and LightGBM (RO1)
`code/04_tabular_tuning.py`. Rolling-origin CV on the training period only: persistence D-1 and D-7, a Ridge alpha grid and 40 random LightGBM configurations.

In [ ]:
"""Step 4: tune the baselines, Ridge and LightGBM with rolling-origin CV on the training period only."""
import lightgbm as lgb
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import Ridge

df = pd.read_pickle(os.path.join(PROC, "tab_observed_popw.pkl"))
FEATS = feature_cols(df)
rows = []

def cv_eval(fit_predict, name, extra=None, store_oof=False, frame=df):
    res, oof = [], []
    for k in range(len(FOLDS)):
        tr, va = split_masks(frame, k)
        t0 = time.time(); p = fit_predict(frame[tr], frame[va]); dt = time.time() - t0
        m = metrics(frame.y[va], p, frame.date[va].values, frame.slot[va].values); m.update(fold=k, fit_s=dt); res.append(m)
        if store_oof: oof.append(pd.DataFrame(dict(date=frame.date[va].values, slot=frame.slot[va].values, y=frame.y[va].values, p=p)))
    r = pd.DataFrame(res); out = dict(model=name, **(extra or {}))
    for c in ["MAE", "RMSE", "MAPE", "R2", "PW_MAE", "TD_MAE", "PeakMAE"]:
        out[c] = r[c].mean(); out[c + "_sd"] = r[c].std()
    out["per_fold_MAE"] = r.MAE.round(1).tolist(); out["fit_s"] = r.fit_s.mean()
    return out, (pd.concat(oof) if store_oof else None)

# ---- naive baselines
for name, col in [("Persistence D-1", "lag_d1"), ("Persistence D-7", "lag_d7")]:
    o, oof = cv_eval(lambda tr, va, c=col: va[c].values, name, store_oof=True); rows.append(o)
    oof.to_pickle(os.path.join(TAB, f"oof_{name.replace(' ', '_')}.pkl"))

# ---- Ridge (linear baseline, one-hot slot and day-of-week, standardised numerics)
cat = ["slot", "dow"]; num = [c for c in FEATS if c not in cat]
def ridge_fp(alpha):
    def fp(tr, va):
        pipe = Pipeline([("pre", ColumnTransformer([("num", StandardScaler(), num), ("cat", OneHotEncoder(handle_unknown="ignore"), cat)])),
                         ("m", Ridge(alpha=alpha))])
        pipe.fit(tr[FEATS], tr.y); return pipe.predict(va[FEATS])
    return fp
ridge_grid = []
for a in np.logspace(-3, 3, 13):
    o, _ = cv_eval(ridge_fp(a), "Ridge", dict(alpha=a)); ridge_grid.append(o)
rg = pd.DataFrame(ridge_grid); rg.to_csv(os.path.join(TAB, "tune_ridge.csv"), index=False)
best_a = rg.loc[rg.MAE.idxmin(), "alpha"]
o, oof = cv_eval(ridge_fp(best_a), "Ridge", dict(alpha=best_a), store_oof=True); rows.append(o); oof.to_pickle(os.path.join(TAB, "oof_Ridge.pkl"))
print("ridge best alpha", best_a, o["MAE"])

# ---- LightGBM random search
rng = np.random.default_rng(SEED)
def sample():
    return dict(num_leaves=int(rng.choice([15, 31, 63, 127])), learning_rate=float(np.exp(rng.uniform(np.log(.01), np.log(.1)))),
                n_estimators=int(rng.choice([300, 600, 1000, 1500])), min_child_samples=int(rng.choice([10, 20, 50, 100])),
                feature_fraction=float(rng.uniform(.6, 1)), bagging_fraction=float(rng.uniform(.6, 1)),
                lambda_l2=float(np.exp(rng.uniform(np.log(1e-3), np.log(10)))), max_depth=int(rng.choice([-1, 6, 10])))
def lgb_fp(params, target="abs", weights="none", seed=SEED, feats=None):
    def fp(tr, va):
        f = feats or FEATS
        ytr, base_tr = make_target(tr, target); _, base_va = make_target(va, target)
        w = make_weights(tr.y.values, weights)
        m = lgb.LGBMRegressor(objective="l1", bagging_freq=1, random_state=seed, n_jobs=1, verbose=-1, **params)
        m.fit(tr[f], ytr, sample_weight=w); return m.predict(va[f]) + base_va
    return fp
search = []
for it in range(40):
    p = sample(); t0 = time.time(); o, _ = cv_eval(lgb_fp(p), "LightGBM", dict(params=p)); search.append(o)
    print(it, round(o["MAE"], 1), round(time.time() - t0, 1), flush=True)
sd = pd.DataFrame(search); sd.to_pickle(os.path.join(TAB, "tune_lgbm.pkl"))
best = sd.loc[sd.MAE.idxmin(), "params"]; json.dump(best, open(os.path.join(TAB, "best_lgbm.json"), "w"), indent=1)
o, oof = cv_eval(lgb_fp(best), "LightGBM", dict(params=best), store_oof=True); rows.append(o); oof.to_pickle(os.path.join(TAB, "oof_LightGBM.pkl"))
pd.DataFrame(rows).to_pickle(os.path.join(TAB, "cv_tabular_main.pkl"))
print(pd.DataFrame(rows)[["model", "MAE", "MAE_sd", "RMSE", "MAPE", "PW_MAE", "TD_MAE", "PeakMAE", "per_fold_MAE"]])

## Step 05: tune the LSTM and Transformer (RO1)
`code/05_deep_tuning.py`. Random search of 12 configurations per architecture on the same six folds.

In [ ]:
"""Step 5: random-search tuning of the LSTM and Transformer on the same rolling-origin folds."""
import pickle
for arch, n_cfg in [("LSTM", 12), ("Transformer", 12)]:   # was: python 05_deep_tuning.py <arch> 12
    d = pickle.load(open(os.path.join(PROC, "daily_observed_popw.pkl"), "rb"))
    rng = np.random.default_rng(SEED + (0 if arch == "LSTM" else 1))
    def sample():
        base = dict(dropout=float(rng.choice([0, .1, .2, .3])), lr=float(np.exp(rng.uniform(np.log(3e-4), np.log(3e-3)))),
                    head_hidden=int(rng.choice([32, 64])), batch=int(rng.choice([16, 32])))
        if arch == "LSTM": base.update(hidden=int(rng.choice([32, 64, 128])), layers=int(rng.choice([1, 2])))
        else: base.update(d_model=int(rng.choice([32, 64])), heads=int(rng.choice([2, 4])), layers=int(rng.choice([1, 2, 3])))
        return base
    rows = []
    for c in range(n_cfg):
        hp = sample(); res = []; t0 = time.time()
        for k in range(len(FOLDS)):
            tr, va = day_masks(d, k); p, info = fit_predict(d, tr, va, arch, hp)
            m = flat_metrics(d, va, p); m.update(fold=k, epochs=info["epochs"], best_epoch=info["best_epoch"], n_params=info["n_params"]); res.append(m)
        r = pd.DataFrame(res)
        rows.append(dict(hp=hp, MAE=r.MAE.mean(), MAE_sd=r.MAE.std(), PW_MAE=r.PW_MAE.mean(), per_fold=r.MAE.round(1).tolist(),
                         epochs=r.epochs.mean(), n_params=int(r.n_params.iloc[0]), sec=time.time() - t0))
        print(c, hp, round(r.MAE.mean(), 1), r.MAE.round(0).tolist(), round(time.time() - t0), flush=True)
        pd.DataFrame(rows).to_pickle(os.path.join(TAB, f"tune_{arch}.pkl"))
    df = pd.DataFrame(rows); best = df.loc[df.MAE.idxmin(), "hp"]
    json.dump(best, open(os.path.join(TAB, f"best_{arch}.json"), "w"), indent=1); print("BEST", best)

## Step 06: LightGBM ablations (RO2, RO3)
`code/06_tabular_ablations.py`. Target and loss weighting, lambda sensitivity, weather information and city proxy, feature groups and training-window length, with the tuned hyperparameters held fixed.

In [ ]:
"""Step 6: LightGBM ablations on the rolling-origin folds (best hyperparameters held fixed)."""
import lightgbm as lgb
best = json.load(open(os.path.join(TAB, "best_lgbm.json")))
frames = {k: pd.read_pickle(os.path.join(PROC, f"tab_{k}.pkl")) for k in ["observed_popw", "persist_popw", "none_popw", "observed_avg3", "observed_akl"]}
def run(frame, target="abs", weights="none", lam=None, drop=(), seeds=(42,), params=None, oof_name=None, train_days=None):
    params = params or best; feats = feature_cols(frame, drop); res, oofs = [], []
    for k in range(len(FOLDS)):
        tr, va = split_masks(frame, k)
        if train_days:   # learning curve: keep only the most recent N training days
            keep = sorted(frame.date[tr].unique())[-train_days:]; tr = tr & frame.date.isin(keep).values
        T, V = frame[tr], frame[va]; ytr, _ = make_target(T, target); _, bva = make_target(V, target)
        w = make_weights(T.y.values, weights, lam=lam); ps = []
        for sd in seeds:
            m = lgb.LGBMRegressor(objective="l1", bagging_freq=1, random_state=sd, n_jobs=1, verbose=-1, **params)
            m.fit(T[feats], ytr, sample_weight=w); p = m.predict(V[feats]) + bva; ps.append(p)
            mm = metrics(V.y, p, V.date.values, V.slot.values); mm.update(fold=k, seed=sd); res.append(mm)
        if oof_name: oofs.append(pd.DataFrame(dict(date=V.date.values, slot=V.slot.values, y=V.y.values, p=np.mean(ps, 0))))
    if oof_name: pd.concat(oofs).to_pickle(os.path.join(TAB, f"oof_{oof_name}.pkl"))
    return pd.DataFrame(res)
out = []
def rec(tag, **kw):
    t0 = time.time(); r = run(**kw); r["exp"] = tag; out.append(r)
    f = r.groupby("fold")[["MAE", "PW_MAE", "TD_MAE"]].mean()
    print(f"{tag:40s} MAE {f.MAE.mean():6.1f}±{f.MAE.std():5.1f} PW {f.PW_MAE.mean():6.1f} TD {f.TD_MAE.mean():6.1f}  {time.time()-t0:5.0f}s", flush=True)
    pd.concat(out).to_pickle(os.path.join(TAB, "ablation_lgbm.pkl"))
F = frames["observed_popw"]
# A: target x weighting (3 bagging seeds)
for t in ["abs", "delta_origin", "delta_seasonal"]:
    for w in ["none", "bansal", "topdecile"]:
        rec(f"A|{t}|{w}", frame=F, target=t, weights=w, seeds=(42, 43, 44), oof_name=f"LightGBM_{t}_{w}")
# B: lambda sensitivity
for lam in [2, 5]:
    for w in ["bansal", "topdecile"]: rec(f"B|lam{lam}|{w}", frame=F, weights=w, lam=lam)
# C: weather information and city proxy
rec("C|weather none", frame=frames["none_popw"]); rec("C|weather persist(D-1)", frame=frames["persist_popw"])
rec("C|city Auckland only", frame=frames["observed_akl"]); rec("C|city 3-city mean", frame=frames["observed_avg3"])
# D: feature groups
for grp in ["lags", "calendar", "weather"]: rec(f"D|drop {grp}", frame=F, drop=(grp,))
# E: learning curve (training-window length)
for n in [90, 180, 270]: rec(f"E|last {n} train days", frame=F, train_days=n)

## Step 07: LSTM and Transformer ablations (RO2)
`code/07_deep_ablations.py`. Target x loss-weighting grid on the rolling-origin folds, two seeds each.

In [ ]:
"""Step 7: LSTM and Transformer target x weighting ablation on the rolling-origin folds (2 seeds)."""
import pickle
for arch in ["LSTM", "Transformer"]:   # was: python 07_deep_ablations.py <arch>
    d = pickle.load(open(os.path.join(PROC, "daily_observed_popw.pkl"), "rb"))
    hp = json.load(open(os.path.join(TAB, f"best_{arch}.json")))
    out, oofs = [], {}
    combos = [(t, w) for t in ["abs", "delta_origin", "delta_seasonal"] for w in ["none", "bansal", "topdecile"]]
    part = ""
    for t, w in combos:
        t0 = time.time(); oo = []
        for k in range(len(FOLDS)):
            tr, va = day_masks(d, k); ps = []
            for sd in (42, 43):
                p, info = fit_predict(d, tr, va, arch, hp, target=t, weights=w, seed=sd); ps.append(p)
                m = flat_metrics(d, va, p); m.update(fold=k, seed=sd, exp=f"A|{t}|{w}", epochs=info["epochs"]); out.append(m)
            pm = np.mean(ps, 0)
            oo.append(pd.DataFrame(dict(date=np.repeat(d["dates"][va], 48), slot=np.tile(np.arange(48), va.sum()), y=d["Y"][va].ravel(), p=pm.ravel())))
        pd.concat(oo).to_pickle(os.path.join(TAB, f"oof_{arch}_{t}_{w}.pkl"))
        r = pd.DataFrame(out); r.to_pickle(os.path.join(TAB, f"ablation_{arch}{part}.pkl"))
        f = r[r.exp == f"A|{t}|{w}"].groupby("fold")[["MAE", "PW_MAE", "TD_MAE"]].mean()
        print(f"{t:15s} {w:10s} MAE {f.MAE.mean():6.1f}±{f.MAE.std():5.1f} PW {f.PW_MAE.mean():6.1f} TD {f.TD_MAE.mean():6.1f} {time.time()-t0:5.0f}s", flush=True)

## Step 11: CV summaries and figures
`code/11_cv_figures.py`. CV summary tables, corrected resampled t-tests and the CV figures. In the scripts this step is launched by step 12, so it runs here before step 12.

In [ ]:
"""Step 11: CV summary tables and figures, per-fold comparison (RO1/RO3) and target x weighting ablation (RO2)."""
import matplotlib.pyplot as plt, pickle
set_style(); out = {}
main = pd.read_pickle(os.path.join(TAB, "cv_tabular_main.pkl"))
abl = {"LightGBM": pd.read_pickle(os.path.join(TAB, "ablation_lgbm.pkl"))}
for a in ["LSTM", "Transformer"]:
    import glob
    ps = sorted(glob.glob(os.path.join(TAB, f"ablation_{a}*.pkl")))
    if ps: abl[a] = pd.concat([pd.read_pickle(p) for p in ps]).drop_duplicates(subset=["exp", "fold", "seed"])
# per-architecture target x weight summary (mean over folds of seed-mean)
summ = []
for arch, r in abl.items():
    r = r[r.exp.str.startswith("A|")]
    for exp, g in r.groupby("exp"):
        f = g.groupby("fold")[["MAE", "RMSE", "MAPE", "R2", "PW_MAE", "TD_MAE", "TD_under", "PeakMAE", "PeakTimeErr_h"]].mean()
        sd_seed = g.groupby("seed").MAE.mean().std() if g.seed.nunique() > 1 else np.nan
        _, t, w = exp.split("|")
        summ.append(dict(arch=arch, target=t, weights=w, **{c: f[c].mean() for c in f.columns}, MAE_fold_sd=f.MAE.std(),
                         PW_fold_sd=f.PW_MAE.std(), TD_fold_sd=f.TD_MAE.std(), MAE_seed_sd=sd_seed, fold_MAE=f.MAE.round(1).tolist(), fold_PW=f.PW_MAE.round(1).tolist(), fold_TD=f.TD_MAE.round(1).tolist()))
S = pd.DataFrame(summ)
for arch in S.arch.unique():
    b = S[(S.arch == arch) & (S.target == "abs") & (S.weights == "none")].iloc[0]
    for c in ["MAE", "PW_MAE", "TD_MAE"]: S.loc[S.arch == arch, c + "_chg%"] = 100 * (S.loc[S.arch == arch, c] / b[c] - 1)
    # how many folds improved PW-MAE vs baseline
    S.loc[S.arch == arch, "folds_PW_better"] = [int(np.sum(np.array(x) < np.array(b.fold_PW))) for x in S.loc[S.arch == arch, "fold_PW"]]
    S.loc[S.arch == arch, "folds_TD_better"] = [int(np.sum(np.array(x) < np.array(b.fold_TD))) for x in S.loc[S.arch == arch, "fold_TD"]]
S.to_pickle(os.path.join(TAB, "ablation_summary.pkl")); print(S.drop(columns=["fold_MAE", "fold_PW", "fold_TD"]).round(1).to_string())
# corrected resampled t-test (Nadeau & Bengio) across the 6 folds for each variant vs abs/none (PW-MAE)
from scipy import stats
ti = json.load(open(os.path.join(TAB, "tabular_info.json")))
n_tr = np.array([f["train_days"] for f in ti["folds"]]); n_te = np.array([f["val_days"] for f in ti["folds"]])
def crt(a, b):
    d = np.array(a) - np.array(b); k = len(d); var = d.var(ddof=1)
    t = d.mean() / np.sqrt((1 / k + np.mean(n_te / n_tr)) * var); return t, 2 * stats.t.sf(abs(t), k - 1)
tests = {}
for arch in S.arch.unique():
    b = S[(S.arch == arch) & (S.target == "abs") & (S.weights == "none")].iloc[0]
    for _, r in S[S.arch == arch].iterrows():
        if r.target == "abs" and r.weights == "none": continue
        tests[f"{arch}|{r.target}|{r.weights}"] = dict(PW=crt(r.fold_PW, b.fold_PW), TD=crt(r.fold_TD, b.fold_TD), MAE=crt(r.fold_MAE, b.fold_MAE))
out["crt_vs_abs"] = tests; save_json(out, "cv_stats.json")
# ---- figure: per-fold MAE of main models (best target of each arch shown as standard abs/none)
fig, ax = plt.subplots(figsize=(6.6, 2.6)); x = np.arange(1, 7)
series = [("Persistence D-1", main[main.model == "Persistence D-1"].per_fold_MAE.iloc[0]), ("Ridge", main[main.model == "Ridge"].per_fold_MAE.iloc[0])]
for arch in S.arch.unique(): series.append((arch, S[(S.arch == arch) & (S.target == "abs") & (S.weights == "none")].fold_MAE.iloc[0]))
for k, (n, v) in enumerate(series): ax.plot(x, v, marker="o", ms=4, color=COLORS[k], label=n)
ax.set_xticks(x, ["F1 Jun–Jul 25\n(winter)", "F2 Aug–Sep", "F3 Oct–Nov", "F4 Dec–Jan", "F5 Feb–Mar", "F6 Apr–May 26"]); ax.set_ylabel("Validation MAE (MW)")
ax.legend(ncol=5, fontsize=7, loc="upper right"); savefig(fig, "fig_cv_folds"); plt.close(fig)
# ---- figure: % change of PW-MAE and TD-MAE vs abs/none
fig, axs = plt.subplots(1, 2, figsize=(6.6, 3.6), sharey=True)
labels = [f"{t.replace('delta_origin','Δ-origin').replace('delta_seasonal','Δ-slot').replace('abs','abs')}\n{w.replace('none','L1').replace('bansal','PW mean').replace('topdecile','PW top10')}" for t in ["abs", "delta_origin", "delta_seasonal"] for w in ["none", "bansal", "topdecile"]][1:]
arches = [a for a in ["LightGBM", "Transformer"] if a in set(S.arch)]; wbar = .8 / len(arches); xx = np.arange(len(labels))
for j, c in enumerate(["PW_MAE_chg%", "TD_MAE_chg%"]):
    for k, arch in enumerate(arches):
        v = [S[(S.arch == arch) & (S.target == t) & (S.weights == w)][c].iloc[0] for t in ["abs", "delta_origin", "delta_seasonal"] for w in ["none", "bansal", "topdecile"]][1:]
        axs[j].barh(xx + k * wbar, v, height=wbar * .9, color=COLORS[k + 2] if False else COLORS[[0, 1, 2][k]], label=arch)
    axs[j].axvline(0, color=MUTED, lw=.8); axs[j].axvline(-10, color=MUTED, lw=.8, ls=":")
    axs[j].set_xlabel(("PW-MAE" if j == 0 else "Top-decile MAE") + " change vs abs/L1 (%)")
axs[0].set_yticks(xx + wbar * (len(arches) - 1) / 2, labels, fontsize=7); axs[0].invert_yaxis()
h, l = axs[0].get_legend_handles_labels(); fig.legend(h, l, ncol=3, loc="upper center", bbox_to_anchor=(0.55, 1.06), fontsize=8)
fig.tight_layout(); savefig(fig, "fig_ablation_target"); plt.close(fig)

## Step 12: fix the selection before testing
`code/12_select.py`. Writes every selection decision to `selection.json` from the CV results only, before the test set is scored.

In [ ]:
"""Step 12: fix every selection decision from the CV results before the test set is scored (writes selection.json)."""
# step 11 (previous cell) has already written ablation_summary.pkl
S = pd.read_pickle(os.path.join(TAB, "ablation_summary.pkl"))
rg = pd.read_csv(os.path.join(TAB, "tune_ridge.csv"))
sel = dict(ridge_alpha=float(rg.loc[rg.MAE.idxmin(), "alpha"]), lgbm_params=json.load(open(os.path.join(TAB, "best_lgbm.json"))),
           rule_variant="per architecture: the target x loss combination with the lowest mean CV PW-MAE (RO2 candidate), plus the abs/L1 reference",
           rule_final="lowest mean CV PW-MAE among all carried variants whose mean CV MAE is within 5% of the best mean CV MAE")
cands = []
for arch in ["LightGBM", "LSTM", "Transformer"]:
    s = S[S.arch == arch]; nb = s[~((s.target == "abs") & (s.weights == "none"))]; b = nb.loc[nb.PW_MAE.idxmin()]
    v = {"abs-L1": ["abs", "none"], f"{b.target}-{b.weights}": [b.target, b.weights]}
    sel[f"{arch}_variants" if arch != "LightGBM" else "lgbm_variants"] = v
    if arch != "LightGBM": sel[f"{arch}_hp"] = json.load(open(os.path.join(TAB, f"best_{arch}.json")))
    for tag, (t, w) in v.items():
        r = s[(s.target == t) & (s.weights == w)].iloc[0]; cands.append(dict(name=f"{arch} {tag}", MAE=r.MAE, PW=r.PW_MAE))
C = pd.DataFrame(cands); ok = C[C.MAE <= 1.05 * C.MAE.min()]; final = ok.loc[ok.PW.idxmin(), "name"]
sel["final_model"] = final; sel["candidates_cv"] = C.round(2).to_dict(orient="records")
arch_f = final.split()[0]
sel["LSTM_main_tag"] = [k for k in sel["LSTM_variants"] if k != "abs-L1"][0] if arch_f != "LSTM" else final.split()[1]
sel["Transformer_main_tag"] = [k for k in sel["Transformer_variants"] if k != "abs-L1"][0] if arch_f != "Transformer" else final.split()[1]
sel["shap_variant"] = [k for k in sel["lgbm_variants"] if k != "abs-L1"][0]
best_of = {a: C[C.name.str.startswith(a)].sort_values("PW").name.iloc[0] for a in ["LightGBM", "LSTM", "Transformer"]}
pairs = [[final, "Persistence D-1"], [final, "Persistence D-7"], [final, "Ridge"]] + [[final, best_of[a]] for a in best_of if best_of[a] != final]
for a in ["LightGBM", "LSTM", "Transformer"]:
    vv = sel["lgbm_variants" if a == "LightGBM" else f"{a}_variants"]; ro2 = [k for k in vv if k != "abs-L1"][0]; pairs.append([f"{a} {ro2}", f"{a} abs-L1"])
sel["dm_pairs"] = pairs; sel["dm_pairs_pw"] = [p for p in pairs[-3:]]
sel["plot_models"] = [final] + [best_of[a] for a in best_of if best_of[a] != final][:1] + ["Ridge"]
sel["decided_at"] = time.strftime("%Y-%m-%d %H:%M:%S %Z")
json.dump(sel, open(os.path.join(TAB, "selection.json"), "w"), indent=1); print(json.dumps(sel, indent=1))

## Step 08: final test (scored once)
`code/08_final_test.py`. Retrains the selected configurations on the full training period with five seeds and scores the held-out winter 2026 test set once. Saves all models.

In [ ]:
"""Step 8: retrain the selected configurations on the full training period
(data from 1 Sep 2024, target days 8 Sep 2024 to 31 May 2026) and score the held-out winter 2026 test set once.
The variants were fixed from the CV results before this script was run (see out/tables/selection.json)."""
import pickle, joblib, lightgbm as lgb
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import Ridge
sel = json.load(open(os.path.join(TAB, "selection.json")))
F = pd.read_pickle(os.path.join(PROC, "tab_observed_popw.pkl")); FEATS = feature_cols(F)
tr, te = split_masks(F, test=True); T, V = F[tr], F[te]
d = pickle.load(open(os.path.join(PROC, "daily_observed_popw.pkl"), "rb")); dtr, dte = day_masks(d, test=True)
assert (np.repeat(d["dates"][dte], 48) == V.date.values).all()
SEEDS = [42, 43, 44, 45, 46]
preds, rows, costs = {}, [], {}
def add(name, p, seed_metrics=None):
    preds[name] = p; m = metrics(V.y, p, V.date.values, V.slot.values); m["model"] = name
    if seed_metrics is not None:
        sm = pd.DataFrame(seed_metrics)
        for c in ["MAE", "RMSE", "MAPE", "PW_MAE", "TD_MAE", "PeakMAE"]: m[c + "_seed_sd"] = sm[c].std(); m[c + "_seed_mean"] = sm[c].mean()
    rows.append(m); print(name, {k: round(v, 2) for k, v in m.items() if k in ["MAE", "RMSE", "MAPE", "PW_MAE", "TD_MAE", "PeakMAE"]}, flush=True)
add("Persistence D-1", V.lag_d1.values); add("Persistence D-7", V.lag_d7.values)
cat = ["slot", "dow"]; num = [c for c in FEATS if c not in cat]
t0 = time.time()
ridge = Pipeline([("pre", ColumnTransformer([("num", StandardScaler(), num), ("cat", OneHotEncoder(handle_unknown="ignore"), cat)])), ("m", Ridge(alpha=sel["ridge_alpha"]))]).fit(T[FEATS], T.y)
costs["Ridge"] = dict(train_s=time.time() - t0); add("Ridge", ridge.predict(V[FEATS]))
joblib.dump(dict(pipeline=ridge, features=FEATS), os.path.join(MOD, "ridge_pipeline.joblib"))
for tag, (t, w) in sel["lgbm_variants"].items():
    ps, sm = [], []; t0 = time.time()
    for sd in SEEDS:
        ytr, _ = make_target(T, t); _, bva = make_target(V, t)
        m = lgb.LGBMRegressor(objective="l1", bagging_freq=1, random_state=sd, n_jobs=1, verbose=-1, **sel["lgbm_params"])
        m.fit(T[FEATS], ytr, sample_weight=make_weights(T.y.values, w)); p = m.predict(V[FEATS]) + bva; ps.append(p)
        sm.append(metrics(V.y, p, V.date.values, V.slot.values))
        if sd == 42: joblib.dump(dict(model=m, features=FEATS, target=t, weights=w), os.path.join(MOD, f"lgbm_{tag}.joblib"))
    costs[f"LightGBM {tag}"] = dict(train_s=(time.time() - t0) / len(SEEDS)); add(f"LightGBM {tag}", np.mean(ps, 0), sm)
for arch in ["LSTM", "Transformer"]:
    hp = sel[f"{arch}_hp"]
    for tag, (t, w) in sel[f"{arch}_variants"].items():
        ps, sm, eps = [], [], []; t0 = time.time()
        for sd in SEEDS:
            p, info, model, sc = fit_predict(d, dtr, dte, arch, hp, target=t, weights=w, seed=sd, return_model=True)
            ps.append(p.ravel()); sm.append(flat_metrics(d, dte, p)); eps.append(info["best_epoch"])
            # every seed's weights are saved, so the five-seed ensemble can be reloaded and averaged at serving time
            torch.save(model.state_dict(), os.path.join(MOD, f"{arch}_{tag}_seed{sd}.pt"))
            if sd == 42:
                torch.save(model.state_dict(), os.path.join(MOD, f"{arch}_{tag}.pt")); pickle.dump(info["history"], open(os.path.join(TAB, f"history_{arch}_{tag}.pkl"), "wb"))
                # the scaler is fitted on the same training days for every seed, so one copy serves the whole ensemble
                pickle.dump(vars(sc), open(os.path.join(MOD, f"{arch}_{tag}_scaler.pkl"), "wb"))
                costs[f"{arch} {tag}"] = dict(n_params=info["n_params"])
        costs[f"{arch} {tag}"].update(train_s=(time.time() - t0) / len(SEEDS), best_epoch_mean=float(np.mean(eps)))
        add(f"{arch} {tag}", np.mean(ps, 0), sm)
res = pd.DataFrame(rows); res.to_pickle(os.path.join(TAB, "test_results.pkl"))
pickle.dump(dict(preds=preds, date=V.date.values, slot=V.slot.values, y=V.y.values), open(os.path.join(TAB, "test_preds.pkl"), "wb"))
save_json(costs, "costs_train.json")
print(res[["model", "MAE", "RMSE", "MAPE", "R2", "PW_MAE", "TD_MAE", "TD_under", "PeakMAE", "PeakTimeErr_h", "PeakUnder", "bias"]].round(2).to_string())

## Step 10: test analysis
`code/10_test_analysis.py`. Diebold-Mariano tests, day-block bootstrap CIs, error analysis, SHAP and permutation importance, latency and model size, seasonal breakdown.

In [ ]:
"""Step 10: statistical tests, error analysis, interpretability and costs on the already scored test predictions."""
import pickle, joblib, glob
import matplotlib.pyplot as plt, lightgbm as lgb, shap
set_style()
sel = json.load(open(os.path.join(TAB, "selection.json")))
TP = pickle.load(open(os.path.join(TAB, "test_preds.pkl"), "rb")); P = TP["preds"]; y = TP["y"]; dates = TP["date"]; slots = TP["slot"]
res = pd.read_pickle(os.path.join(TAB, "test_results.pkl")); out = {}
FINAL = sel["final_model"]
# ---- bootstrap CIs (day blocks) for MAE and TD-MAE
mae = lambda a, b: np.mean(np.abs(a - b))
def tdmae(a, b): t = a >= np.quantile(a, .9); return np.mean(np.abs(a[t] - b[t]))
ci = {}
for m in P:
    ci[m] = dict(MAE=block_bootstrap_ci(dates, y, P[m], mae).tolist(), TD_MAE=block_bootstrap_ci(dates, y, P[m], tdmae, n=500).tolist())
out["ci"] = ci
# ---- Diebold-Mariano on daily MAE series
dm = {}
for a in P:
    for b in P:
        if a >= b: continue
        ea, eb = daily_mae(dates, y, P[a]).values, daily_mae(dates, y, P[b]).values
        s, p = dm_test(ea, eb); dm[f"{a} | {b}"] = dict(stat=s, p=p, mean_diff=float(np.mean(ea - eb)))
# DM on daily top-decile-weighted error (peak-focused)
def daily_pw(m):
    thr = np.quantile(y, .9); w = np.where(y >= thr, PW_LAMBDA, 1.0)
    return pd.DataFrame(dict(d=dates, e=w * np.abs(y - P[m]))).groupby("d").e.mean().values
dmpw = {}
for a, b in sel["dm_pairs_pw"]:
    s, p = dm_test(daily_pw(a), daily_pw(b)); dmpw[f"{a} | {b}"] = dict(stat=s, p=p, mean_diff=float(np.mean(daily_pw(a) - daily_pw(b))))
out["dm"] = dm; out["dm_pw"] = dmpw
# ---- error analysis of final model
df = pd.DataFrame(dict(date=dates, slot=slots, y=y, p=P[FINAL])); df["e"] = df.y - df.p; df["ae"] = df.e.abs()
g = load_grid(); temp = g.groupby("date").temp_popw.mean(); hol = g.groupby("date").holiday.first()
dd = df.groupby("date").agg(MAE=("ae", "mean"), bias=("e", "mean"), ymax=("y", "max")).join(temp.rename("temp")).join(hol.rename("holiday"))
dd["dow"] = pd.to_datetime(dd.index).day_name()
out["worst_days"] = dd.sort_values("MAE", ascending=False).head(8).round(1).reset_index().to_dict(orient="records")
out["corr_dailyMAE_temp"] = float(dd.MAE.corr(dd.temp)); out["corr_dailyMAE_ymax"] = float(dd.MAE.corr(dd.ymax))
out["mae_by_dow"] = df.assign(dow=pd.to_datetime(df.date).dt.day_name()).groupby("dow").ae.mean().round(1).to_dict()
out["mae_holidays"] = float(df[df.date.map(hol) == 1].ae.mean()) if (df.date.map(hol) == 1).any() else None
cold = df[(df.date >= "2026-08-04") & (df.date <= "2026-08-07")]
out["coldsnap"] = {m: dict(MAE=float(np.mean(np.abs(cold.y - pd.Series(P[m], index=df.index)[cold.index]))),
                          max_pred=float(pd.Series(P[m], index=df.index)[cold.index].max())) for m in P}
out["coldsnap_actual_max"] = float(cold.y.max())
out["train_max"] = float(load_grid().query("date < @TEST_START").mw.max())
for m in P: out.setdefault("test_max_pred", {})[m] = float(np.max(P[m]))
# ---- figures
names = sel["plot_models"]
fig, ax = plt.subplots(figsize=(6.6, 2.6))
wk = df[(df.date >= "2026-08-02") & (df.date <= "2026-08-09")]; x = pd.to_datetime(wk.date) + pd.to_timedelta(wk.slot * 30, unit="min")
ax.plot(x, wk.y, color=INK, lw=1.6, label="Actual")
for k, m in enumerate(names): ax.plot(x, pd.Series(P[m], index=df.index)[wk.index], color=COLORS[k], lw=1.1, label=pretty(m))
ax.axhline(out["train_max"], color=MUTED, ls="--", lw=.8); ax.text(x.iloc[-60], out["train_max"] + 40, "training maximum (6,705 MW)", fontsize=7, color=MUTED)
ax.set_ylabel("Demand (MW)"); ax.set_ylim(3300, 7900); ax.legend(ncol=4, loc="upper left", fontsize=7)
import matplotlib.dates as mdates; ax.xaxis.set_major_formatter(mdates.DateFormatter("%a %d %b")); savefig(fig, "fig_test_coldsnap"); plt.close(fig)
fig, axs = plt.subplots(1, 2, figsize=(6.6, 2.6))
for k, m in enumerate(names):
    e = pd.DataFrame(dict(s=slots, ae=np.abs(y - P[m]))).groupby("s").ae.mean(); axs[0].plot(np.arange(48) / 2, e.values, color=COLORS[k], label=pretty(m))
axs[0].set_xlabel("Local time (h)"); axs[0].set_ylabel("Test MAE (MW)"); axs[0].set_xticks(range(0, 25, 6)); axs[0].legend(fontsize=7)
a = df.pivot(index="date", columns="slot", values="y").max(1)
for k, m in enumerate(names):
    b = pd.DataFrame(dict(d=dates, s=slots, p=P[m])).pivot(index="d", columns="s", values="p").max(1)
    axs[1].scatter(a, b, s=9, color=COLORS[k], label=pretty(m), alpha=.8)
lo, hi = a.min() - 100, a.max() + 100; axs[1].plot([lo, hi], [lo, hi], color=MUTED, lw=.8, ls="--")
axs[1].set_xlabel("Actual daily peak (MW)"); axs[1].set_ylabel("Forecast daily peak (MW)")
fig.tight_layout(); savefig(fig, "fig_test_slot_peak"); plt.close(fig)
# ---- SHAP for the companion LightGBM (shap_variant in selection.json, seed 42), not the selected Transformer
F = pd.read_pickle(os.path.join(PROC, "tab_observed_popw.pkl")); V = F[F.date >= TEST_START]
art = joblib.load(os.path.join(MOD, f"lgbm_{sel['shap_variant']}.joblib")); feats = art["features"]
ex = shap.TreeExplainer(art["model"]); sv = ex.shap_values(V[feats])
imp = pd.Series(np.abs(sv).mean(0), index=feats).sort_values(); out["shap_mean_abs"] = imp.sort_values(ascending=False).round(1).to_dict()
fig, ax = plt.subplots(figsize=(4.6, 3.6)); top = imp.tail(15)
ax.barh(top.index, top.values, color=COLORS[0], height=.6); ax.set_xlabel("Mean |SHAP value| on test (MW of target)"); savefig(fig, "fig_shap"); plt.close(fig)
# group permutation importance on the held-out test winter, same companion LightGBM
base = mae(V.y.values, art["model"].predict(V[feats]) + make_target(V, art["target"])[1]); rng = np.random.default_rng(SEED); perm = {}
for gname, cols in FEATURE_GROUPS.items():
    cols = [c for c in cols if c in feats]; deltas = []
    for r in range(5):
        Vp = V.copy(); idx = rng.permutation(len(V))
        for c in cols: Vp[c] = V[c].values[idx]
        deltas.append(mae(V.y.values, art["model"].predict(Vp[feats]) + make_target(Vp, art["target"])[1]) - base)
    perm[gname] = [float(np.mean(deltas)), float(np.std(deltas))]
out["perm_group_importance"] = perm
# ---- costs: inference latency for one day-ahead forecast (48 values) and model size
lat = {}
X1 = V[feats].iloc[:48]
t0 = time.perf_counter(); [art["model"].predict(X1) for _ in range(200)]; lat["LightGBM"] = (time.perf_counter() - t0) / 200 * 1000
rd = joblib.load(os.path.join(MOD, "ridge_pipeline.joblib")); t0 = time.perf_counter(); [rd["pipeline"].predict(X1) for _ in range(200)]; lat["Ridge"] = (time.perf_counter() - t0) / 200 * 1000
import torch
d = pickle.load(open(os.path.join(PROC, "daily_observed_popw.pkl"), "rb")); tr, te = day_masks(d, test=True)
sc = Scaler().fit(d, tr); H, Fu = sc.transform(d, te); H1, F1 = torch.tensor(H[:1]), torch.tensor(Fu[:1])
for arch in ["LSTM", "Transformer"]:
    tag = sel[f"{arch}_main_tag"]; mdl = ARCH[arch](H.shape[-1], Fu.shape[-1], **sel[f"{arch}_hp"]); mdl.load_state_dict(torch.load(os.path.join(MOD, f"{arch}_{tag}.pt"))); mdl.eval()
    with torch.no_grad():
        t0 = time.perf_counter(); [mdl(H1, F1) for _ in range(200)]; lat[arch] = (time.perf_counter() - t0) / 200 * 1000
out["latency_ms"] = lat
out["model_size_kb"] = {os.path.basename(f): os.path.getsize(f) / 1024 for f in glob.glob(os.path.join(MOD, "*"))}
# ---- RO3: seasonal breakdown from CV out-of-fold predictions and test
season = {}
for f in sorted(glob.glob(os.path.join(TAB, "oof_*.pkl"))):
    o = pd.read_pickle(f); name = os.path.basename(f)[4:-4]; m = pd.to_datetime(o.date).dt.month
    w = m.isin([6, 7, 8])
    season[name] = dict(winter_MAE=float(np.mean(np.abs(o.y - o.p)[w])), nonwinter_MAE=float(np.mean(np.abs(o.y - o.p)[~w])),
                        winter_TD=metrics(o.y[w], o.p[w])["TD_MAE"], nonwinter_TD=metrics(o.y[~w], o.p[~w])["TD_MAE"])
out["season_oof"] = season
save_json(out, "test_analysis.json"); print(json.dumps({k: v for k, v in out.items() if k not in ("dm", "season_oof", "model_size_kb")}, indent=1, default=float)[:6000])

## Step 13: learning and training curves
`code/13_extra_figures.py`. Learning curve over training-window length and the neural-network training curves.

In [ ]:
"""Learning curve (training-window length) and neural-network training curves."""
import pickle, matplotlib.pyplot as plt
set_style()
r = pd.read_pickle(os.path.join(TAB, "ablation_lgbm.pkl")); ti = json.load(open(os.path.join(TAB, "tabular_info.json")))
pts = []
for e, n in [("E|last 90 train days", 90), ("E|last 180 train days", 180), ("E|last 270 train days", 270), ("A|abs|none", None)]:
    q = r[r.exp == e].groupby("fold")[["MAE", "TD_MAE"]].mean()
    pts.append((n or np.mean([f["train_days"] for f in ti["folds"]]), q.MAE.mean(), q.MAE.std(), q.TD_MAE.mean()))
pts = np.array(pts)
fig, axs = plt.subplots(1, 2, figsize=(6.6, 2.5))
axs[0].errorbar(pts[:, 0], pts[:, 1], yerr=pts[:, 2], marker="o", ms=4, color=COLORS[0], capsize=3, label="MAE (± fold SD)")
axs[0].plot(pts[:, 0], pts[:, 3], marker="s", ms=4, color=COLORS[1], label="Top-decile MAE")
axs[0].set_xlabel("Training days used (most recent)"); axs[0].set_ylabel("CV error (MW)"); axs[0].legend(fontsize=7)
sel = json.load(open(os.path.join(TAB, "selection.json")))
for k, (arch, tag) in enumerate([("LSTM", sel["LSTM_main_tag"]), ("Transformer", sel["Transformer_main_tag"])]):
    h = np.array(pickle.load(open(os.path.join(TAB, f"history_{arch}_{tag}.pkl"), "rb")))
    axs[1].plot(np.arange(1, len(h) + 1), h[:, 0], color=COLORS[k], lw=1, ls="--", label=f"{arch} train")
    axs[1].plot(np.arange(1, len(h) + 1), h[:, 1], color=COLORS[k], lw=1.5, label=f"{arch} early-stop set")
axs[1].set_xlabel("Epoch"); axs[1].set_ylabel("Weighted L1 (scaled)"); axs[1].legend(fontsize=7)
fig.tight_layout(); savefig(fig, "fig_learning"); plt.close(fig)
print(pts.round(1))

## Step 09: pipeline diagram
`code/09_pipeline_figure.py`. Draws the pipeline figure (Figure 3.1).

In [ ]:
"""Pipeline diagram (Figure 3.1)."""
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch
set_style()
fig, ax = plt.subplots(figsize=(6.6, 3.3)); ax.axis("off"); ax.set_xlim(0, 10); ax.set_ylim(0, 5)
def box(x, y, w, h, title, body, c):
    ax.add_patch(FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.02,rounding_size=0.08", fc="white", ec=c, lw=1.4))
    ax.text(x + w / 2, y + h - .17, title, ha="center", va="top", fontsize=8, weight="bold", color=INK)
    ax.text(x + w / 2, y + h - .48, body, ha="center", va="top", fontsize=6.1, color=MUTED, linespacing=1.25)
def arrow(x1, y1, x2, y2): ax.annotate("", (x2, y2), (x1, y1), arrowprops=dict(arrowstyle="->", color=MUTED, lw=1))
box(0.1, 3.1, 2.2, 1.7, "Raw data", "24 EMI Grid_export files\n(175,210 rows, 223 POCs)\n3 Open-Meteo city files\nNZ holiday calendar", COLORS[0])
box(2.7, 3.1, 2.2, 1.7, "01 Build series", "sum POCs, kWh/500 → MW\nTP → true UTC → 48-slot\nlocal grid (DST fix)\nweather TZ alignment", COLORS[0])
box(5.3, 3.1, 2.2, 1.7, "02–03 EDA + features", "re-run EDA on clean data\n25 features (lags, calendar,\nweather), origin = end of d−1\n3 target formulations", COLORS[0])
box(7.9, 3.1, 2.0, 1.7, "Chronological split", "train/CV: 8 Sep 2024–\n31 May 2026 (631 d)\ntest: Jun–Aug 2026\n(92 d, used once)", COLORS[1])
box(0.1, 0.4, 3.0, 2.1, "04–05 Tune (RO1)", "6 expanding rolling-origin folds\nPersistence D-1 / D-7\nRidge (α grid)\nLightGBM (40 random configs)\nLSTM, Transformer (12 each)", COLORS[2])
box(3.5, 0.4, 3.0, 2.1, "06–07 Ablations (RO2, RO3)", "target: abs / Δ-origin / Δ-slot\nloss: L1 / PW-MAE (mean, top-10%)\nλ = 2, 3, 5; weather none /\npersist / oracle; city proxy;\nfeature groups; train length", COLORS[2])
box(6.9, 0.4, 3.0, 2.1, "08 Final test (once)", "retrain on all training days\n5 seeds; score winter 2026\nDM tests, block bootstrap CI\nerror analysis, SHAP, costs\nsaved pipelines (joblib/.pt)", COLORS[1])
arrow(2.3, 3.95, 2.7, 3.95); arrow(4.9, 3.95, 5.3, 3.95); arrow(7.5, 3.95, 7.9, 3.95)
arrow(6.4, 3.1, 1.6, 2.5); arrow(6.4, 3.1, 5.0, 2.5); arrow(3.1, 1.45, 3.5, 1.45); arrow(6.5, 1.45, 6.9, 1.45); arrow(8.9, 3.1, 8.4, 2.5)
savefig(fig, "fig_pipeline")

## Step 15: forecasting-service diagram
`code/15_service_figure.py`. Draws the proposed service architecture for the application conversion section.

In [ ]:
"""Proposed forecasting-service architecture (application conversion figure)."""
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch
set_style()
fig, ax = plt.subplots(figsize=(6.6, 3.0)); ax.axis("off"); ax.set_xlim(0, 10); ax.set_ylim(0, 4.6)
def box(x, y, w, h, t, b, c):
    ax.add_patch(FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.02,rounding_size=0.08", fc="white", ec=c, lw=1.3))
    ax.text(x + w / 2, y + h - .15, t, ha="center", va="top", fontsize=7.6, weight="bold", color=INK)
    ax.text(x + w / 2, y + h - .45, b, ha="center", va="top", fontsize=6.3, color=MUTED, linespacing=1.25)
def arr(a, b, c, d): ax.annotate("", (c, d), (a, b), arrowprops=dict(arrowstyle="->", color=MUTED, lw=1))
box(0.1, 2.6, 2.1, 1.8, "Inputs (daily, 23:30)", "metered GXP demand d−1\n(EMI / operator SCADA)\nweather forecast day d\n(3 cities, hourly)\nholiday calendar", COLORS[0])
box(2.6, 2.6, 2.3, 1.8, "Validation + features", "schema & range checks\nDST-aware 48-slot grid\n25 features, same code\nas training (common.py)", COLORS[0])
# size of one saved Transformer checkpoint, read from the run rather than typed in
tag = json.load(open(os.path.join(TAB, "selection.json")))["Transformer_main_tag"]
ckpt = os.path.join(MOD, f"Transformer_{tag}.pt")
size = f"{os.path.getsize(ckpt) / 1024:.0f} KB each" if os.path.exists(ckpt) else "saved per seed"
box(5.3, 2.6, 2.1, 1.8, "Inference", f"saved pipeline:\nscalers + Transformer\n(5 seeds, {size})\n< 1 ms per forecast", COLORS[2])
box(7.8, 2.6, 2.1, 1.8, "Delivery", "REST API (JSON, 48 MW)\ndashboard for operators\nCSV to market systems\npeak alert if > threshold", COLORS[2])
box(1.2, 0.2, 3.4, 1.8, "Monitoring", "daily MAE, TD-MAE, peak error\nvs persistence & Ridge fallbacks\ninput drift: PSI on demand,\ntemperature; alert on breach", COLORS[1])
box(5.4, 0.2, 3.4, 1.8, "Retraining", "monthly expanding-window refit\n+ early retrain on drift alert\nchampion–challenger on last\n8 weeks before promotion", COLORS[1])
arr(2.2, 3.5, 2.6, 3.5); arr(4.9, 3.5, 5.3, 3.5); arr(7.4, 3.5, 7.8, 3.5); arr(8.8, 2.6, 4.0, 2.0); arr(4.6, 1.1, 5.4, 1.1); arr(7.1, 2.0, 6.35, 2.6)
savefig(fig, "fig_service")

## Step 14: numbers quoted in the report
`code/14_vals.py`. Collects the quoted numbers into `report/vals_auto.json`.

In [ ]:
"""Collect the numbers quoted in the report into report/vals_auto.json, keyed by name."""
import glob, pickle, sklearn, lightgbm, torch, statsmodels, shap, matplotlib
V = {}
a = json.load(open(os.path.join(TAB, "data_audit.json"))); e = json.load(open(os.path.join(TAB, "eda.json"))); ti = json.load(open(os.path.join(TAB, "tabular_info.json")))
V.update(v_pandas=pd.__version__, v_numpy=np.__version__, v_sklearn=sklearn.__version__, v_lgbm=lightgbm.__version__, v_torch=torch.__version__.split("+")[0],
         v_statsmodels=statsmodels.__version__, v_shap=shap.__version__, v_mpl=matplotlib.__version__)
V.update(raw_rows=f"{a['raw_rows']:,}", raw_cols=a["raw_cols"], pocs=a["pocs"], pocs_min=a["pocs_per_month_min"], pocs_max=a["pocs_per_month_max"], days=a["days"],
         hh_true=f"{a['halfhours_true_time']:,}", interp=a["slots_interpolated_spring_forward"], grid_rows=f"{a['grid_rows']:,}",
         r_hourly=f"{e['r_temp_popw']:.2f}".replace("-", "−"), r_daily=f"{e['r_daily_mean_temp_popw']:.2f}".replace("-", "−"), r_naive=f"{e['r_temp_naive_A1_alignment']:.2f}".replace("-", "−"),
         tiwai_mean=f"{e['tiwai_mean']:.0f}", tiwai_share=f"{100*e['tiwai_share']:.1f}", tiwai_low_days=e["tiwai_days_below_450"],
         n_days=ti["days"], n_rows=f"{ti['rows']:,}", n_feat=ti["n_features"], acf1=f"{e['acf']['1']:.2f}", acf48=f"{e['acf']['48']:.2f}", acf96=f"{e['acf']['96']:.2f}", acf336=f"{e['acf']['336']:.2f}",
         wkday=f"{e['weekday_mean']:,.0f}", wkend=f"{e['weekend_mean']:,.0f}", peak_slot=e["modal_peak_slot"], evening_share=f"{100*e['peak_slot_share_evening']:.0f}",
         peak_winter_share=f"{100*e['share_peak_in_winter_top_decile']:.0f}", train_days=ti["train_days"], test_days=ti["test_days"], test_rows=f"{ti['test_days']*48:,}",
         winter_mean=f"{e['winter_mean']:,.0f}", nonwinter_mean=f"{e['nonwinter_mean']:,.0f}", train_mean=f"{e['by_split']['train']['mean']:,.0f}", test_mean=f"{e['by_split']['test']['mean']:,.0f}",
         yoy1=f"{e['yoy']['Sep24-Aug25']['mean']:,.0f}", yoy2=f"{e['yoy']['Sep25-Aug26']['mean']:,.0f}", yoymax1=f"{e['yoy']['Sep24-Aug25']['max']:,.0f}", yoymax2=f"{e['yoy']['Sep25-Aug26']['max']:,.0f}")
# tuning times (sum of per-config seconds; machine shared by two jobs)
tl = pd.read_pickle(os.path.join(TAB, "tune_lgbm.pkl")); ls = pd.read_pickle(os.path.join(TAB, "tune_LSTM.pkl")); tf = pd.read_pickle(os.path.join(TAB, "tune_Transformer.pkl"))
lg_sec = tl.fit_s.sum() * 6
hm = lambda s: f"{s/3600:.1f} h" if s > 3600 else f"{s/60:.0f} min"
vt = dict(t_ridge="<1 min", t_lgbm=hm(lg_sec), t_lstm=hm(ls.sec.sum()), t_tf=hm(tf.sec.sum()))
save_json(vt, "vals_tuning.json"); V["dl_cost_ratio"] = f"{(ls.sec.mean()+tf.sec.mean())/2/(lg_sec/len(tl)):.0f}"
V["total_runtime"] = "about 3 hours on an Apple-silicon MacBook Pro, CPU only (about 8 hours on a 2-vCPU Linux machine); dominated by LSTM tuning and ablations"
os.makedirs(os.path.join(ROOT, "report"), exist_ok=True)
json.dump(V, open(os.path.join(ROOT, "report", "vals_auto.json"), "w"), indent=1, default=str); print(V); print(vt)

## Step 16: weather sensitivity and K-means day types
`code/16_weather_and_clusters.py`. Post-hoc: retrains the selected Transformer with persistence weather, and clusters training-day demand profiles with K-means (unsupervised). Changes no selection decision.

In [ ]:
"""Step 16: post-hoc sensitivity analyses, run after model selection (they change no selection decision).

A. Weather sensitivity of the selected model. The selected Transformer (target, loss weighting and tuned
   hyperparameters read from selection.json) is retrained on the training period with five seeds, once with
   observed target-day weather (as in the main results) and once with persistence weather (the same slot of
   day D-1, which is known at the forecast origin). Both are scored on the winter 2026 test set.
B. Unsupervised day-type analysis. K-means clusters the standardised 48-slot demand profiles of the training
   days (scaler and clusters fitted on training days only, k chosen by silhouette score). Test days are assigned
   to the nearest centroid, and the forecast error of both weather variants is broken down by cluster and by
   distance to the nearest centroid (how unusual a day is compared with the training data).
"""
import pickle
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler
from scipy.stats import spearmanr

sel = json.load(open(os.path.join(TAB, "selection.json")))
hp = sel["Transformer_hp"]
TARGET, WEIGHTS = sel["Transformer_variants"][sel["Transformer_main_tag"]]   # the selected Transformer variant
SEEDS = [42, 43, 44, 45, 46]
g = load_grid()
out = {}

# ---- A. weather sensitivity of the selected Transformer
preds = {}
for weather in ["observed", "persist"]:
    d = build_daily(g, weather=weather)
    dtr, dte = day_masks(d, test=True)
    ps, single = [], []
    t0 = time.time()
    for sd in SEEDS:
        p, info = fit_predict(d, dtr, dte, "Transformer", hp, target=TARGET, weights=WEIGHTS, seed=sd)
        ps.append(p); single.append(flat_metrics(d, dte, p)["MAE"])
        print(weather, sd, round(single[-1], 1), info["best_epoch"], flush=True)
    p = np.mean(ps, 0); preds[weather] = p
    m = flat_metrics(d, dte, p)
    out[weather] = dict(MAE=m["MAE"], RMSE=m["RMSE"], MAPE=m["MAPE"], PW_MAE=m["PW_MAE"], TD_MAE=m["TD_MAE"],
                        PeakMAE=m["PeakMAE"], single_seed_mean=float(np.mean(single)), single_seed_sd=float(np.std(single, ddof=1)),
                        minutes=(time.time() - t0) / 60)
    print(weather, {k: round(v, 2) for k, v in out[weather].items()}, flush=True)

Y = d["Y"][dte]; dates_te = d["dates"][dte]
dmae = {w: np.abs(Y - preds[w]).mean(1) for w in preds}
s, pval = dm_test(dmae["persist"], dmae["observed"])
out["dm_persist_vs_observed"] = dict(delta=float((dmae["persist"] - dmae["observed"]).mean()), DM=float(s), p=float(pval))
out["persist_gap_pct"] = 100 * (out["persist"]["MAE"] / out["observed"]["MAE"] - 1)

# ---- B. unsupervised day types (K-means on training-day demand profiles)
P_tr, P_te = d["Y"][dtr], Y
zs = StandardScaler().fit(P_tr)
Z_tr, Z_te = zs.transform(P_tr), zs.transform(P_te)
sil = {}
for k in range(2, 9):
    km = KMeans(n_clusters=k, n_init=20, random_state=SEED).fit(Z_tr)
    sil[k] = float(silhouette_score(Z_tr, km.labels_))
k_best = max(sil, key=sil.get)
km = KMeans(n_clusters=k_best, n_init=20, random_state=SEED).fit(Z_tr)
lab_tr, lab_te = km.labels_, km.predict(Z_te)
dist_tr = np.min(km.transform(Z_tr), 1); dist_te = np.min(km.transform(Z_te), 1)
dates_tr = pd.to_datetime(d["dates"][dtr]); dte_ts = pd.to_datetime(dates_te)
rows = []
for c in range(k_best):
    mtr, mte = lab_tr == c, lab_te == c
    rows.append(dict(cluster=c, train_days=int(mtr.sum()), test_days=int(mte.sum()),
                     mean_mw=float(P_tr[mtr].mean()) if mtr.any() else np.nan,
                     peak_mw=float(P_tr[mtr].max(1).mean()) if mtr.any() else np.nan,
                     weekend_share=float((dates_tr[mtr].dayofweek >= 5).mean()) if mtr.any() else np.nan,
                     winter_share=float(dates_tr[mtr].month.isin([6, 7, 8]).mean()) if mtr.any() else np.nan,
                     test_mae_observed=float(dmae["observed"][mte].mean()) if mte.any() else np.nan,
                     test_mae_persist=float(dmae["persist"][mte].mean()) if mte.any() else np.nan))
out["kmeans"] = dict(k=k_best, silhouette=sil, clusters=rows,
                     novelty_threshold_p95=float(np.quantile(dist_tr, .95)),
                     test_days_beyond_p95=int((dist_te > np.quantile(dist_tr, .95)).sum()),
                     spearman_dist_vs_mae_observed=[float(x) for x in spearmanr(dist_te, dmae["observed"])],
                     spearman_dist_vs_mae_persist=[float(x) for x in spearmanr(dist_te, dmae["persist"])])
far = dist_te > np.quantile(dist_tr, .95)
out["kmeans"]["mae_far_vs_near"] = dict(observed=[float(dmae["observed"][far].mean()), float(dmae["observed"][~far].mean())],
                                         persist=[float(dmae["persist"][far].mean()), float(dmae["persist"][~far].mean())])
cold = (dte_ts >= "2026-08-04") & (dte_ts <= "2026-08-07")
out["kmeans"]["cold_spell"] = dict(clusters=[int(x) for x in lab_te[cold]], beyond_p95=[bool(x) for x in far[cold]],
                                    mae_observed=float(dmae["observed"][cold].mean()), mae_persist=float(dmae["persist"][cold].mean()))
pickle.dump(dict(preds=preds, dates=dates_te, y=Y, labels_test=lab_te, dist_test=dist_te), open(os.path.join(TAB, "weather_clusters_preds.pkl"), "wb"))
save_json(out, "weather_and_clusters.json")
print(json.dumps(out, indent=1, default=float))

## Step 17: supporting checks
`code/17_report_checks.py`. Target SDs, tuning cost ratios, peak timing, Holm correction and Newey-West DM checks. Reads saved outputs only.

In [ ]:
"""Step 17: supporting numbers quoted in the report that no earlier step writes out.

Reads only saved outputs (no model is retrained) and writes out/tables/report_checks.json:
  A. target standard deviations on the training days (absolute, delta-to-origin, delta-to-same-slot),
  B. tuning cost per configuration for LightGBM, LSTM and Transformer, and the ratios between them,
  C. daily peak-timing comparison of the selected Transformer with the absolute-target Transformer on the test winter,
  D. Holm correction across the nine Diebold-Mariano tests reported in Table 4.5,
  E. the same nine DM tests with Newey-West (Bartlett) variances using 1 to 3 lags of daily autocovariance.
"""
from scipy import stats
import pickle

sel = json.load(open(os.path.join(TAB, "selection.json")))
out = {}

# ---- A. target variance on training days (same targets as make_target)
F = pd.read_pickle(os.path.join(PROC, "tab_observed_popw.pkl"))
tr = F[(F.date >= FIRST_TARGET_DAY) & (F.date < TEST_START)]
sd = {k: float(np.std(make_target(tr, k)[0], ddof=1)) for k in ["abs", "delta_origin", "delta_seasonal"]}
out["target_sd_train"] = {k: round(v, 1) for k, v in sd.items()}
out["delta_seasonal_variance_share"] = round((sd["delta_seasonal"] / sd["abs"]) ** 2, 3)

# ---- B. tuning cost per configuration (LightGBM fit_s is per fold, so x6 folds)
tl = pd.read_pickle(os.path.join(TAB, "tune_lgbm.pkl"))
lg = float(tl.fit_s.mean() * len(FOLDS))
cost = {"LightGBM": lg}
for arch in ["LSTM", "Transformer"]:
    cost[arch] = float(pd.read_pickle(os.path.join(TAB, f"tune_{arch}.pkl")).sec.mean())
out["tuning_sec_per_config"] = {k: round(v, 1) for k, v in cost.items()}
out["tuning_cost_ratio_vs_lgbm"] = {k: round(cost[k] / lg, 1) for k in ["LSTM", "Transformer"]}

# ---- C. daily peak timing: selected Transformer vs absolute-target Transformer
P = pickle.load(open(os.path.join(TAB, "test_preds.pkl"), "rb"))
df = pd.DataFrame(dict(d=pd.to_datetime(P["date"]), s=np.asarray(P["slot"]), y=P["y"]))
A = df.pivot(index="d", columns="s", values="y").values
ta = A.argmax(1)
models = {"selected": sel["final_model"], "abs": "Transformer abs-L1"}
err = {}
pk = {}
for key, name in models.items():
    B = pd.DataFrame(dict(d=df.d, s=df.s, p=P["preds"][name])).pivot(index="d", columns="s", values="p").values
    tb = B.argmax(1)
    e = np.abs(ta - tb) / 2                      # hours (48 half-hour slots)
    swap = (ta < 24) != (tb < 24)               # peak placed in the wrong half of the day (morning vs evening)
    err[key] = e
    pk[key] = dict(model=name, mean_error_h=round(float(e.mean()), 2), exact_days=int((e == 0).sum()), days=int(len(e)),
                   am_pm_swap_days=int(swap.sum()), share_of_error_from_swaps=round(float(e[swap].sum() / e.sum()), 3))
s, p = dm_test(err["selected"], err["abs"])
pk["dm_selected_vs_abs"] = dict(DM=round(float(s), 2), p=round(float(p), 3))
out["peak_timing"] = pk

# ---- D. Holm correction over the Table 4.5 family (p-values from step 10)
TA = json.load(open(os.path.join(TAB, "test_analysis.json")))
def dm_p(table, a, b):
    for k in (f"{a} | {b}", f"{b} | {a}"):
        if k in TA[table]: return float(TA[table][k]["p"])
    raise KeyError((a, b))
S = sel["final_model"]
LG, LS, TA_ = "LightGBM delta_seasonal-none", "LSTM delta_seasonal-none", "Transformer abs-L1"
fam = [("dm", S, "Persistence D-1"), ("dm", S, "Persistence D-7"), ("dm", S, "Ridge"), ("dm", S, LG), ("dm", S, LS), ("dm", S, TA_),
       ("dm_pw", S, TA_), ("dm", LG, "LightGBM abs-L1"), ("dm", LS, "LSTM abs-L1")]
raw = [dm_p(t, a, b) for t, a, b in fam]
order = np.argsort(raw); m = len(raw); adj = [0.0] * m; run = 0.0
for rank, i in enumerate(order):
    run = max(run, min(1.0, raw[i] * (m - rank))); adj[i] = run
out["holm_table_4_5"] = [dict(test=t, a=a, b=b, p=round(r, 4), p_holm=round(q, 4), significant_holm=bool(q < 0.05))
                         for (t, a, b), r, q in zip(fam, raw, adj)]

# ---- E. DM with Newey-West variance (lags 1 to 3) on the daily MAE series
pw_w = np.where(df.y.values >= np.quantile(df.y.values, .9), PW_LAMBDA, 1.0)   # same peak weights as daily_pw in step 10
def daily(name, weighted=False):
    w = pw_w if weighted else 1.0
    return pd.DataFrame(dict(d=df.d, e=w * np.abs(df.y.values - P["preds"][name]))).groupby("d").e.mean().values
def dm_nw(e1, e2, L):
    x = e1 - e2; n = len(x); c = x - x.mean()
    g = [np.sum(c[k:] * c[:n - k]) / n for k in range(L + 1)]
    v = g[0] + 2 * sum((1 - k / (L + 1)) * g[k] for k in range(1, L + 1))
    s = x.mean() / np.sqrt(v / n) * np.sqrt((n - 1) / n)          # same small-sample factor as dm_test with h = 1
    return float(2 * stats.t.sf(abs(s), n - 1))
nw = []
for t, a, b in fam:
    ea, eb = daily(a, t == "dm_pw"), daily(b, t == "dm_pw"); c = (ea - eb) - (ea - eb).mean()
    ps = {f"lag{L}": round(dm_nw(ea, eb, L), 4) for L in (0, 1, 2, 3)}
    nw.append(dict(test=t, a=a, b=b, lag1_autocorr=round(float(np.corrcoef(c[1:], c[:-1])[0, 1]), 2), **ps,
                   same_decision_at_5pct=len({p < 0.05 for p in ps.values()}) == 1))
out["dm_newey_west"] = nw

save_json(out, "report_checks.json")
print(json.dumps(out, indent=1))